# GZ-CMD passo a passo: a fila de revisão manual abaixo do ponto de corte

Material didático para profissionais e estudantes de saúde coletiva, sem pré-requisito de estatística ou de aprendizado de máquina. O tema é o *linkage* (relacionamento de registros) entre o Sistema de Informações sobre Mortalidade (SIM) e o Sistema de Informação de Agravos de Notificação (Sinan), nas notificações de tuberculose.

O notebook demonstra o GZ-CMD na configuração recomendada pela tese: **preservar os vínculos aceitos pelo ponto de corte do OpenRecLink e, abaixo dele, ordenar a fila de revisão manual pela probabilidade calibrada fora da amostra**, isolada ou fundida ao escore do OpenRecLink, escolhendo pela curva de esforço. A automação das decisões com um modelo de linguagem de grande porte (LLM) aparece como contraste, e o perfil histórico do pacote, como linha de base.

> **Dados sintéticos.** Nenhum registro real, nome, endereço ou data. A base imita a *dificuldade* do problema da tese, não os seus números: o que se vê aqui demonstra o método e não é evidência sobre ele.

**Como ler.** Cada célula de código vem depois de um texto curto que diz o que ela faz e por quê. Depois das saídas importantes, uma célula **Leitura** ajuda a interpretar. Os números citados nas leituras são impressos pelo próprio código, e a função `confere` testa as frases do texto que dependem deles. Os números da tese aparecem só na seção 10, como referência citada.

| Seção | Pergunta |
|---|---|
| 0. Glossário | O que quer dizer cada termo? |
| 1. O problema | Por que ordenar a revisão manual abaixo do ponto de corte? |
| 2. Teoria do GZ-CMD | Como a política de quatro ações escolhe entre aceitar, rejeitar, revisar ou consultar o LLM? |
| 3. Dados sintéticos | O que a base simula, e por quê? |
| 4. Probabilidade calibrada fora da amostra | Como estimar, sem vazamento, a probabilidade de cada potencial par ser par verdadeiro? |
| 5. Ordenações da fila | Quatro maneiras de ordenar a fila: O1, O2, O3 e F4. |
| 6. Curva de esforço | Quantas revisões cada fila exige para cada meta? |
| 7. Incerteza | A diferença entre filas se sustenta no *bootstrap*? |
| 8. Contraste com a automação | A política com LLM recupera mais que a fila, na mesma carga humana? |
| 9. Linha de base histórica | O perfil histórico do pacote continua reproduzível? |
| 10. Fluxo recomendado e limitações | O que a tese recomenda e o que este notebook não mostra. |

A execução completa leva de um a três minutos num computador portátil; a seção 4 concentra o tempo.

## Preparação do ambiente

Execute esta parte primeiro, sempre.

- **No Google Colab**, a célula abaixo baixa o repositório público do pacote e instala a biblioteca `gzcmd_record_linkage`. Na primeira vez, leva cerca de um minuto.
- **No computador local**, com o repositório já clonado, ela só localiza a pasta do projeto e ajusta os caminhos de importação.

In [ ]:
# Preparação do ambiente: torna o notebook executável LOCALMENTE e no GOOGLE COLAB.
#
# - Localmente (repositório já clonado): localiza a raiz do repositório procurando
#   o `pyproject.toml` nas pastas acima e ajusta o `sys.path`.
# - No Google Colab (só o .ipynb aberto, sem o repositório): clona o repositório e
#   instala a biblioteca. Assim ficam disponíveis a biblioteca
#   `gzcmd_record_linkage`, os módulos do notebook (`synthetic_linkage`, `nb_fila`,
#   `nb_helpers`, `synthetic_data`) e a pasta `data/synthetic/`.
import subprocess
import sys
from pathlib import Path

# Repositório público com a biblioteca, os módulos do notebook e os dados.
REPO_URL = "https://github.com/marco-jardim/gzcmd-record-linkage.git"


def _find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return None


REPO_ROOT = _find_repo_root(Path.cwd().resolve())

if REPO_ROOT is None:
    if "google.colab" not in sys.modules:
        raise RuntimeError(
            "Raiz do repositório não encontrada. Abra o notebook dentro do "
            "repositório clonado; instalação automática só está disponível no Colab."
        )
    # Não há repositório no disco (cenário típico do Google Colab). Clonamos e
    # instalamos o pacote. A primeira execução leva cerca de 1 min.
    _dest = Path.cwd() / "gzcmd-record-linkage"
    if not (_dest / "pyproject.toml").exists():
        print(f"Clonando {REPO_URL} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(_dest)], check=True
        )
    else:
        # Uma sessão anterior pode ter deixado uma cópia antiga: atualizamos,
        # a menos que você tenha editado arquivos dela.
        _editados = subprocess.run(
            ["git", "-C", str(_dest), "status", "--porcelain", "--untracked-files=no"],
            capture_output=True, text=True, check=True,
        ).stdout.strip()
        if _editados:
            print(
                "Aviso: a cópia em gzcmd-record-linkage tem arquivos editados e NÃO foi "
                "atualizada. Apague a pasta para baixar a versão atual."
            )
        else:
            print("Atualizando a cópia já clonada ...")
            subprocess.run(["git", "-C", str(_dest), "fetch", "--depth", "1", "origin"], check=True)
            subprocess.run(["git", "-C", str(_dest), "reset", "--hard", "FETCH_HEAD"], check=True)
    print("Instalando a biblioteca gzcmd_record_linkage (pip)...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", str(_dest)], check=True
    )
    REPO_ROOT = _dest

NB_DIR = REPO_ROOT / "notebooks"
for _path in (str(REPO_ROOT / "src"), str(NB_DIR)):
    if _path in sys.path:
        sys.path.remove(_path)
    sys.path.insert(0, _path)

# Se esta sessão já importou versões antigas dos módulos, esquecemos essas
# cópias para que os imports a seguir leiam os arquivos atuais do disco.
import importlib

_MODULOS_DO_NOTEBOOK = {"nb_helpers", "nb_fila", "synthetic_data", "synthetic_linkage"}
for _modulo in list(sys.modules):
    if _modulo in _MODULOS_DO_NOTEBOOK or _modulo.split(".")[0] == "gzcmd_record_linkage":
        del sys.modules[_modulo]
importlib.invalidate_caches()

DATA_DIR = REPO_ROOT / "data" / "synthetic"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Raiz do repositório: {REPO_ROOT}")
print(f"Pasta de dados.....: {DATA_DIR}")

**Bibliotecas e funções de apoio.** Importamos as bibliotecas de cálculo e de gráficos e três módulos do notebook: `synthetic_linkage` (gera a base sintética), `nb_fila` (probabilidade calibrada, filas, curva de esforço, *bootstrap* e política com LLM simulado) e `nb_helpers` (controles interativos e métricas de apoio). A semente fixa (`SEED`) faz os sorteios se repetirem: quem executar de novo obtém os mesmos números.

Algumas funções curtas formatam números no padrão brasileiro, com vírgula decimal e ponto nos milhares (`br`, `inteiro`, `pct`, e `mostrar` para tabelas). A função `confere` testa uma frase do texto com os números desta execução: se a frase deixar de valer, a execução para e avisa, em vez de deixar o texto desatualizado.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

import nb_fila as nf
import nb_helpers as nbh
import synthetic_linkage as sl
from gzcmd_record_linkage import __version__ as gzcmd_version
from gzcmd_record_linkage.config import load_config

_faltando = [n for n in ("politica_v31", "mesma_carga", "bootstrap_diferenca") if not hasattr(nf, n)]
_faltando += [n for n in ("interagir", "reliability_points") if not hasattr(nbh, n)]
if _faltando:
    raise ImportError(
        f"Módulos de apoio desatualizados: faltam {_faltando}. "
        "Rode de novo a célula de preparação acima ou reinicie a sessão."
    )

SEED = 42
CONFIG_PATH = str(REPO_ROOT / "src" / "gzcmd_record_linkage" / "gzcmd_v3_config.yaml")
cfg = load_config(CONFIG_PATH)
NOMES_MODO = {"vigilancia": "vigilância", "confirmacao": "confirmação"}


def br(x, casas=3):
    # Vírgula decimal e ponto nos milhares: br(12345.678, 1) dá "12.345,7".
    texto = f"{float(x):,.{casas}f}"
    return texto.replace(",", "_").replace(".", ",").replace("_", ".")


def inteiro(n):
    # Inteiro com ponto nos milhares: 30000 vira "30.000".
    return f"{int(n):,}".replace(",", ".")


def pct(x, casas=1):
    # Proporção como porcentagem: 0.123 vira "12,3%".
    return f"{br(100 * x, casas)}%"


def _celula(valor, casas):
    # Formata um valor de tabela no padrão brasileiro.
    if valor is None or valor is pd.NA:
        return ""
    if isinstance(valor, (bool, np.bool_)):
        return "sim" if valor else "não"
    if isinstance(valor, (int, np.integer)):
        return inteiro(valor)
    if isinstance(valor, (float, np.floating)):
        return "" if np.isnan(valor) else br(valor, casas)
    return valor


def mostrar(tabela, casas=3):
    # Exibe uma tabela com vírgula decimal e ponto separando milhares.
    if isinstance(tabela, pd.Series):
        tabela = tabela.to_frame()
    display(tabela.map(lambda valor: _celula(valor, casas)))


def confere(frase, condicao):
    # Confere uma frase do texto com os números desta execução.
    if not bool(condicao):
        raise AssertionError(
            f"O texto afirma: '{frase}', mas esta execução não confirma. Revise o texto."
        )
    print(f"✓ conferido nesta execução: {frase}")


pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda v: br(v, 3))
print(f"gzcmd_record_linkage v{gzcmd_version} | semente = {SEED}")

## 0. Glossário

A terminologia segue a tese. Cada termo é definido de novo, em linguagem simples, na primeira vez que aparece no texto.

**O problema e os dados**

| Termo | O que significa aqui |
|---|---|
| *linkage* (relacionamento de registros) | Método que identifica registros da mesma pessoa em bases de dados distintas. Aqui, óbitos do SIM e notificações de tuberculose do Sinan. |
| blocagem (*blocking*) | Etapa que decide quais registros comparar: só os que compartilham uma chave, como o código fonético do último nome e o sexo. Cada chave é um **passo** (6, 9 e 11 na base sintética). |
| potenciais pares | Pares candidatos formados na blocagem: um óbito (`REFREC`) e uma notificação (`COMPREC`). |
| OpenRecLink | Programa de *linkage* probabilístico usado na tese. Soma pesos de concordância dos campos e dá um escore a cada potencial par. |
| escore | Valor que resume a concordância do potencial par no OpenRecLink: quanto maior, mais parecidos os registros. Não é uma probabilidade. |
| ponto de corte | Valor do escore a partir do qual o OpenRecLink aceita o potencial par como vínculo. Há um por passo. |
| *grey area* (área cinza) | Região abaixo do ponto de corte do OpenRecLink: potenciais pares que o corte não aceitou e entre os quais ainda há pares verdadeiros. |
| Comparador de Registros (CdR) | Programa que compara os campos de identificação em detalhe e produz 29 subescores e a nota final. |
| subescores | As 29 comparações do CdR: fragmentos do nome e do nome da mãe, padrões de erro na data de nascimento, município e endereço. |
| nota final | Resumo dos subescores calculado pelo CdR, de 0 a 10. Neste notebook, "nota" só se usa para o CdR. |
| faixa intermediária da nota final | Notas finais de 5 a 9, com o 5 incluído e o 9 excluído. Abaixo dela, a guarda rejeita sem conta de custos; de 9 para cima, a guarda só aceita quando p ≥ 0,9, e o restante segue para a comparação de custos (seção 2.3). |
| referência operacional | O campo `PAR`. PAR = 1: aceito pelo ponto de corte. PAR = 2: par verdadeiro abaixo do corte recuperado pela revisão manual de origem. PAR = 0: os demais. |
| par verdadeiro, não par | Classe do potencial par segundo a referência operacional: PAR igual a 1 ou 2, ou PAR = 0. "Verdadeiro" remete a essa referência, não a uma verdade absoluta. |
| mesma pessoa (verdade da simulação) | O que só a base sintética sabe: se os dois registros são, de fato, da mesma pessoa. Serve para explicar o mecanismo da simulação (seção 3) e nunca é chamada de par verdadeiro. |
| vínculo | Resultado de uma decisão de *linkage*: o par aceito pelo corte, pela política ou pelo revisor humano. Nunca designa a classe da referência. |
| revisão manual | Conferência de um potencial par pelo revisor humano, um profissional do serviço. A **revisão manual de origem** é a do processo original, que produziu PAR = 2. |
| revisor perfeito simulado | Suposição das contagens deste notebook e da tese: todo par verdadeiro revisado é recuperado, e nenhum não par vira vínculo. |
| falso positivo, falso negativo | Aceitar como vínculo um não par; deixar de vincular um par verdadeiro. |
| sensibilidade | Proporção dos pares verdadeiros que o processo recupera. |
| valor preditivo positivo (VPP) | Proporção de pares verdadeiros entre os potenciais pares aceitos como vínculo. |

**O método**

| Termo | O que significa aqui |
|---|---|
| GZ-CMD | O pós-processamento do *linkage* implementado no pacote `gzcmd_record_linkage`: probabilidade calibrada, ordenação da fila de revisão manual, curva de esforço e as políticas de decisão v3 (histórica, três ações) e v3.1 (quatro ações). |
| probabilidade calibrada | Probabilidade p de o potencial par ser par verdadeiro em que, entre muitos potenciais pares com p perto de 20%, cerca de 20 em cada 100 são pares verdadeiros. |
| fora da amostra | Estimada por um modelo que não viu aquele potencial par, nem o óbito dele, no ajuste. |
| XGBoost | Algoritmo de aprendizado de máquina que soma muitas árvores de decisão pequenas, cada uma corrigindo os erros das anteriores. |
| *Platt scaling* | Regressão logística de uma variável (aqui, a nota final) que a transforma em probabilidade; é a calibração do perfil histórico (seção 9). |
| AUC | Área sob a curva ROC: probabilidade de um par verdadeiro sorteado ficar acima de um não par sorteado. 0,5 é o acaso; 1 é a separação perfeita. |
| precisão média (AP) | Percorra a fila do topo; a cada par verdadeiro encontrado, anote a proporção de pares verdadeiros entre os potenciais pares revisados até ali; a AP é a média dessas anotações. Resume a curva precisão-sensibilidade (*precision-recall*), valoriza o topo da fila e, com pares verdadeiros raros, distingue as ordenações melhor que a AUC. |
| fila: O1, O2, O3, F4 | Ordem de revisão manual da *grey area*: pelo escore (O1), pela nota final (O2), pela probabilidade calibrada (O3) ou pela fusão de O3 com O1 (F4). |
| RRF (fusão por posição recíproca) | Fusão de filas que soma, para cada potencial par, 1/(k + posição) em cada fila. |
| curva de esforço | Número de pares verdadeiros recuperados em função do número de revisões manuais, seguindo uma fila. |
| meta | Número de PAR = 2 a recuperar; aqui, 25%, 64% e 100% dos PAR = 2 da base. |
| orçamento humano | Número de revisões manuais que o serviço consegue fazer. É ele que fixa o ponto de parada na curva de esforço. |
| *bootstrap* agrupado, IC 95% | Reamostragem dos óbitos, com reposição, para medir a incerteza. O intervalo de confiança de 95% vai do percentil 2,5% ao 97,5%. |
| `MATCH`, `NONMATCH`, `DEFERRED`, `LLM_REVIEW` | Ações da política: aceitar como vínculo, rejeitar, enviar à revisão manual, consultar o LLM. |
| *expected loss* (custo esperado de cada ação) | Custo médio que se espera pagar ao tomar uma ação sem saber a classe do potencial par. No texto, "custo esperado". |
| guardas | Regras fixas aplicadas antes da conta de custos. |
| modos | Vigilância (o falso negativo custa mais; prioriza a sensibilidade) e confirmação (o falso positivo custa mais; prioriza o VPP). |
| LLM | Modelo de linguagem de grande porte. Aqui é **simulado** com as taxas de erro do revisor de referência da tese (Claude Haiku 5.5); nenhuma chamada real é feita. |
| MACD (Medidas Contínuas de Diferença de Datas) | Atributos que comparam datas em dias. Na tese, entram na fila F3. Não entram na probabilidade calibrada deste notebook; o perfil histórico da seção 9 os calcula no carregamento (`macd_enabled=True`, o padrão do `run_v3`). |
| *post hoc* | Análise definida depois de ver os resultados. |

## 1. O problema: o que fica abaixo do ponto de corte

No *linkage* probabilístico, o OpenRecLink compara cada potencial par e soma pesos de concordância dos campos num escore. Acima do ponto de corte, o par é aceito como vínculo. Abaixo dele fica a *grey area* (área cinza): a imensa maioria ali é de não pares, mas há pares verdadeiros, em geral de registros com o nome escrito de outro jeito, a data de nascimento ausente ou o endereço incompleto.

Recuperar esses pares exige revisão manual, que consome tempo de profissionais do serviço. A *grey area* tem dezenas de milhares de potenciais pares para poucas dezenas de pares verdadeiros: revisar tudo é inviável, e parar cedo numa fila mal ordenada perde óbitos que deveriam ser vinculados à notificação de tuberculose.

A pergunta prática passa a ser **em que ordem revisar**. Se a fila puser os pares verdadeiros no começo, o mesmo orçamento humano recupera mais. Este notebook compara quatro ordens e ilustra, numa base sintética, por que a tese recomenda ordenar pela probabilidade calibrada, preservando os vínculos que o corte já aceitou.

**Uma fila de brinquedo.** Antes dos dados, a ideia em miniatura: 20 potenciais pares na *grey area*, dos quais 3 são pares verdadeiros. A fila A põe os pares verdadeiros nas posições 2, 5 e 15; a fila B, nas posições 9, 14 e 19. A célula conta, revisão a revisão, quantos pares verdadeiros cada fila já recuperou. Essa contagem acumulada é a **curva de esforço**, que reaparece na seção 6 com a base inteira.

In [ ]:
POSICOES_BRINQUEDO = {"fila A": [2, 5, 15], "fila B": [9, 14, 19]}
N_BRINQUEDO = 20
curvas_brinquedo = {}
for nome, posicoes_verdadeiros in POSICOES_BRINQUEDO.items():
    rotulos = np.zeros(N_BRINQUEDO, dtype=int)
    rotulos[np.asarray(posicoes_verdadeiros) - 1] = 1  # a posição 1 é o começo da fila
    curvas_brinquedo[nome] = nf.curva_esforco(np.arange(N_BRINQUEDO), rotulos)

fig, ax = plt.subplots(figsize=(7, 3.5))
for nome, curva in curvas_brinquedo.items():
    ax.step(curva.index, curva.to_numpy(), where="post", label=nome)
ax.set_title("Fila de brinquedo: pares verdadeiros recuperados a cada revisão")
ax.set_xlabel("Revisões manuais feitas (posição na fila)")
ax.set_ylabel("Pares verdadeiros recuperados")
ax.set_xticks(range(0, N_BRINQUEDO + 1, 2))
ax.legend()
plt.show()

for nome, curva in curvas_brinquedo.items():
    total = int(curva.iloc[-1])
    print(f"{nome}: {nf.recuperados_com_carga(curva, 5)} par(es) verdadeiro(s) com 5 revisões; "
          f"os {total} com {nf.revisoes_para_meta(curva, total)} revisões.")

**Leitura.** As duas filas têm os mesmos pares verdadeiros e, revisadas até o fim, recuperam todos. A diferença está no caminho: a fila A encontra a maior parte nas primeiras revisões. Se o serviço só puder fazer 5 revisões, a ordem decide quantos óbitos serão vinculados à notificação. Nas próximas seções, a mesma ideia numa base com dezenas de milhares de potenciais pares.

## 2. Teoria do GZ-CMD: a política de quatro ações

A tese avaliou uma política que decide sozinha parte dos potenciais pares abaixo do corte, com quatro ações possíveis. Ela **não reduziu a revisão manual além da ordenação da fila**, e a seção 8 refaz essa comparação nesta base. A política vem primeiro porque introduz as duas ideias que o resto do notebook usa: a probabilidade calibrada e o custo dos erros.

Para cada potencial par, a política parte de uma **probabilidade calibrada** $p$ de o par ser par verdadeiro e escolhe a ação de menor **custo esperado** (o *expected loss* do glossário): o custo que se espera pagar, em média, ao tomar aquela ação sem saber a classe do par.

### 2.1 Por que a probabilidade precisa ser calibrada

Uma probabilidade é calibrada quando, entre muitos potenciais pares com $p$ perto de 20%, cerca de 20 em cada 100 são pares verdadeiros. O escore do OpenRecLink e a nota final do CdR ordenam os pares, mas não são probabilidades: um escore alto não diz quantos em cada 100 são pares verdadeiros. Como os custos esperados multiplicam $p$ pelos custos dos erros, uma probabilidade exagerada, ou tímida demais, leva a decisões erradas mesmo com custos bem escolhidos.

### 2.2 As quatro ações e seus custos esperados

$C_{FP}$ é o custo de um falso positivo (aceitar como vínculo um não par) e $C_{FN}$, o de um falso negativo (rejeitar um par verdadeiro). Todos os custos usam a mesma unidade, revisões humanas equivalentes:

| Ação | Código | O que faz | Custo esperado |
|---|---|---|---|
| Aceitar | `MATCH` | aceita o potencial par como vínculo | $M = (1-p)\,C_{FP}$ |
| Rejeitar | `NONMATCH` | descarta o potencial par | $N = p\,C_{FN}$ |
| Revisão manual | `DEFERRED` | envia ao revisor humano | $H = C_H + p\,e^{h}_{FN}\,C_{FN} + (1-p)\,e^{h}_{FP}\,C_{FP}$ |
| Consulta ao LLM | `LLM_REVIEW` | pergunta ao LLM | $L = C_{LLM} + p\,e_{FN}\,C_{FN} + (1-p)\,e_{FP}\,C_{FP} + u\,H$ |

Lendo linha a linha:

- **Aceitar** só dá prejuízo se o par for não par, o que acontece com probabilidade $1-p$.
- **Rejeitar** só dá prejuízo se o par for verdadeiro, com probabilidade $p$.
- **Revisão manual** custa $C_H$ (uma revisão) mais os erros que ainda sobram: o revisor humano deixa passar um par verdadeiro com taxa $e^{h}_{FN}$ e aceita um não par com taxa $e^{h}_{FP}$. Aqui as duas taxas são um erro humano hipotético, igual para os dois lados.
- **Consulta ao LLM** custa $C_{LLM}$, muito menos que uma revisão, mais os erros do LLM ($e_{FN}$ e $e_{FP}$). Quando o LLM responde que não sabe (`UNSURE`), o que acontece com taxa $u$, o par segue ao revisor humano e paga também $H$.

As taxas do LLM dependem da faixa da nota final e do modo. Vêm das contagens do revisor de referência da tese (Claude Haiku 5.5), resumidas no módulo `nb_fila`.

**A regra de decisão.** A ação escolhida é a de menor custo esperado. Em caso de empate, a ordem de preferência é `MATCH`, `NONMATCH`, `DEFERRED` e, por último, `LLM_REVIEW`: decidir sem gastar vem antes, e o LLM só é chamado quando seu custo é estritamente menor que os outros.

### 2.3 Guardas, orçamentos e modos

**Guardas** são regras fixas aplicadas antes da conta (perfil v3.1 do pacote):

- nota final abaixo de 5: `NONMATCH`, sem conta;
- nota final de 9 ou mais **e** $p \ge 0{,}9$: `MATCH`, sem conta;
- o restante segue para a comparação dos quatro custos esperados: a **faixa intermediária da nota final** (de 5 a 9, sem incluir o 9) e as notas finais altas com p abaixo de 0,9.

**Orçamentos.** O LLM tem um número máximo de chamadas; quando ele acaba, o par recebe a melhor ação sem LLM. O orçamento humano é o número de revisões manuais que o serviço consegue fazer. Na política deste notebook ele é ilimitado, e a comparação com as filas é feita depois, na mesma carga humana (seção 8).

**Modos.** Os custos mudam com o objetivo do serviço:

- **vigilância**: perder um óbito com tuberculose notificada (falso negativo) custa mais; prioriza a sensibilidade;
- **confirmação**: unir pessoas diferentes (falso positivo) custa mais; prioriza o valor preditivo positivo (VPP).

A célula seguinte mostra os custos usados na tese, lidos do módulo `nb_fila` em vez de copiados à mão.

In [ ]:
custos = pd.DataFrame(
    {
        NOMES_MODO[modo]: {
            "C_FP (falso positivo)": c_fp,
            "C_FN (falso negativo)": c_fn,
            "C_H (revisão manual)": nf.C_H,
            "C_LLM (consulta ao LLM)": nf.C_LLM,
            "erro humano hipotético": nf.E_HUMANO,
        }
        for modo, (c_fp, c_fn) in nf.CUSTOS_POR_MODO.items()
    }
)
mostrar(custos)
for modo, (c_fp, c_fn) in nf.CUSTOS_POR_MODO.items():
    if c_fn >= c_fp:
        print(f"{NOMES_MODO[modo]}: um falso negativo custa {br(c_fn / c_fp, 1)} vezes um falso positivo.")
    else:
        print(f"{NOMES_MODO[modo]}: um falso positivo custa {br(c_fp / c_fn, 1)} vezes um falso negativo.")
print(f"Uma consulta ao LLM custa {br(nf.C_LLM / nf.C_H, 3)} do custo de uma revisão manual.")
c_fp_vig, c_fn_vig = nf.CUSTOS_POR_MODO["vigilancia"]
c_fp_conf, c_fn_conf = nf.CUSTOS_POR_MODO["confirmacao"]
confere("os dois modos invertem a importância relativa dos erros",
        c_fn_vig > c_fp_vig and c_fp_conf > c_fn_conf)
confere("a consulta ao LLM custa menos de um décimo de uma revisão manual", nf.C_LLM < 0.1 * nf.C_H)

**Leitura.** Os dois modos usam a mesma evidência e invertem a importância relativa dos erros. A consulta ao LLM custa uma pequena fração de uma revisão manual: por isso a política tende a preferi-la quando o LLM erra pouco naquela faixa.

A célula seguinte mostra essas taxas de erro, por modo e faixa da nota final. O motor divide a nota final nas faixas `low`, `grey_low`, `grey_mid`, `grey_high`, `near_high` e `high`. Os nomes `grey_low`, `grey_mid`, `grey_high` e `near_high` são nomes históricos do código para trechos da faixa intermediária da nota final (de 5 a 9); não se referem à *grey area*, que é a região abaixo do ponto de corte. Nas figuras, as faixas aparecem pelos seus limites de nota final.

In [ ]:
taxas_llm = pd.concat(
    {NOMES_MODO[modo]: pd.DataFrame(nf.TAXAS_LLM_HAIKU_V4[modo]).T for modo in nf.TAXAS_LLM_HAIKU_V4},
    names=["modo", "faixa"],
)
mostrar(taxas_llm.rename(columns={"e_fp": "e_FP", "e_fn": "e_FN", "e_unsure": "u (UNSURE)"}))

**Leitura.** Em cada faixa, $e_{FP}$ é a proporção de não pares que o LLM aceitaria e $e_{FN}$, a de pares verdadeiros que ele rejeitaria. Faixas sem decisões do revisor de referência usam valores de reserva do pacote; algumas faixas têm taxas estimadas com poucos pares verdadeiros e, por isso, instáveis. São parâmetros da simulação, não medidas de um LLM nesta base.

### 2.4 Um exemplo resolvido

A função `custos_esperados_v31` escreve os quatro custos da seção 2.2 em código, para qualquer valor de p, e aponta a ação de menor custo (com a ordem de desempate `MATCH`, `NONMATCH`, `DEFERRED`, `LLM_REVIEW`). O argumento `modo_taxas` permite usar os custos de um modo com as taxas do LLM de outro, para separar o efeito de cada um.

In [ ]:
ACOES_V31 = ["MATCH", "NONMATCH", "DEFERRED", "LLM_REVIEW"]


def custos_esperados_v31(p, modo, faixa, modo_taxas=None):
    # Os quatro custos esperados da política v3.1 para um vetor de probabilidades p.
    # modo_taxas: modo de onde vêm as taxas do LLM (padrão: o próprio modo).
    p = np.asarray(p, dtype=float)
    c_fp, c_fn = nf.CUSTOS_POR_MODO[modo]
    taxa = nf.TAXAS_LLM_HAIKU_V4[modo_taxas or modo][faixa]
    custo_m = (1 - p) * c_fp
    custo_n = p * c_fn
    custo_h = nf.C_H + p * nf.E_HUMANO * c_fn + (1 - p) * nf.E_HUMANO * c_fp
    custo_l = (nf.C_LLM + p * taxa["e_fn"] * c_fn + (1 - p) * taxa["e_fp"] * c_fp
               + taxa["e_unsure"] * custo_h)
    tabela = pd.DataFrame(dict(zip(ACOES_V31, [custo_m, custo_n, custo_h, custo_l], strict=True)))
    # argmin devolve a primeira coluna em caso de empate: ordem M, N, H, L.
    tabela["ação de menor custo"] = np.asarray(ACOES_V31)[tabela[ACOES_V31].to_numpy().argmin(axis=1)]
    return tabela


print("Função custos_esperados_v31 pronta.")

Tome um potencial par com probabilidade calibrada de 80%, na faixa `near_high` (nota final de 8 a 9), em que o revisor de referência da tese tem decisões nos dois modos. A célula mostra as taxas do LLM nessa faixa e calcula os quatro custos em três cenários: vigilância, confirmação e, para separar o efeito dos custos do efeito das taxas, confirmação com as taxas da vigilância.

In [ ]:
P_EXEMPLO, FAIXA_EXEMPLO, NOTA_EXEMPLO = 0.80, "near_high", 8.5
assert all(FAIXA_EXEMPLO in contagens for contagens in nf.CONTAGENS_LLM_HAIKU_V4.values()), (
    "a faixa do exemplo precisa ter decisões do revisor de referência nos dois modos"
)
taxas_exemplo = pd.DataFrame(
    {NOMES_MODO[modo]: nf.TAXAS_LLM_HAIKU_V4[modo][FAIXA_EXEMPLO] for modo in nf.CUSTOS_POR_MODO}
)
mostrar(taxas_exemplo)

CENARIOS_EXEMPLO = {
    "vigilância": ("vigilancia", "vigilancia"),
    "confirmação": ("confirmacao", "confirmacao"),
    "confirmação com as taxas da vigilância": ("confirmacao", "vigilancia"),
}
exemplo = pd.concat(
    {nome: custos_esperados_v31([P_EXEMPLO], modo, FAIXA_EXEMPLO, modo_taxas)
     for nome, (modo, modo_taxas) in CENARIOS_EXEMPLO.items()},
    names=["cenário", "linha"],
).droplevel("linha")
mostrar(exemplo)
acao_exemplo = exemplo["ação de menor custo"]
for nome, linha in exemplo.iterrows():
    print(f"{nome}: menor custo = {linha['ação de menor custo']} "
          f"({br(linha[linha['ação de menor custo']], 3)} revisões equivalentes).")

confere("as taxas do LLM nesta faixa são parecidas nos dois modos (diferença menor que 0,1 em cada taxa)",
        (taxas_exemplo.max(axis=1) - taxas_exemplo.min(axis=1) < 0.1).all())
confere("só a troca de custos muda a ação",
        acao_exemplo["vigilância"] != acao_exemplo["confirmação"]
        and acao_exemplo["confirmação com as taxas da vigilância"] == acao_exemplo["confirmação"])
confere("na vigilância, aceitar (MATCH) sai mais barato", acao_exemplo["vigilância"] == "MATCH")
confere("na confirmação, a política não aceita e manda à revisão manual (DEFERRED)",
        acao_exemplo["confirmação"] == "DEFERRED")
confere("na confirmação, o falso positivo custa cinco vezes o falso negativo", c_fp_conf == 5 * c_fn_conf)

**Conferência com o motor do pacote.** As contas acima foram escritas à mão. A célula a seguir monta o motor real (`PolicyEngineV3`, perfil v3.1) para cada cenário, com os mesmos custos e taxas, dá a ele o mesmo potencial par e confere que os quatro custos e a ação coincidem. É esse mesmo motor que a seção 8 usa na base inteira.

In [ ]:
from gzcmd_record_linkage.policy_engine import (
    Budget,
    Costs,
    HumanBudget,
    HumanError,
    LLMError,
    PolicyEngineV3,
)

COLUNAS_DO_MOTOR = {"MATCH": "loss_match", "NONMATCH": "loss_nonmatch",
                    "DEFERRED": "loss_human", "LLM_REVIEW": "loss_llm"}
for nome, (modo, modo_taxas) in CENARIOS_EXEMPLO.items():
    c_fp, c_fn = nf.CUSTOS_POR_MODO[modo]
    motor = PolicyEngineV3(
        Costs(false_positive=c_fp, false_negative=c_fn, llm_review=nf.C_LLM, human_review=nf.C_H),
        {faixa: LLMError(**taxa) for faixa, taxa in nf.TAXAS_LLM_HAIKU_V4[modo_taxas].items()},
        Budget(llm_max=1),  # uma consulta basta: o exemplo tem um só potencial par
        policy_version="v3.1",
        human_error=HumanError(nf.E_HUMANO, nf.E_HUMANO),
        human_budget=HumanBudget(None),
    )
    par = pd.DataFrame({"nota final": [NOTA_EXEMPLO], "p_cal": [P_EXEMPLO], "band": [FAIXA_EXEMPLO]})
    saida = motor.triage(par).iloc[0]
    for acao_motor, coluna in COLUNAS_DO_MOTOR.items():
        assert np.isclose(saida[coluna], exemplo.loc[nome, acao_motor]), (nome, acao_motor)
    assert saida["action"] == exemplo.loc[nome, "ação de menor custo"], nome
    print(f"✓ {nome}: os quatro custos e a ação ({saida['action']}) coincidem com o motor do pacote.")

**Leitura.** A probabilidade é a mesma, e as taxas do LLM nesta faixa são parecidas nos dois modos; o que decide é o custo dos erros. Na vigilância, aceitar sai mais barato. Na confirmação, em que o falso positivo custa cinco vezes o falso negativo, a política não aceita e manda à revisão manual. A linha da confirmação com as taxas da vigilância mostra que trocar só as taxas do LLM não muda a ação: a evidência sobre o par (p) fica separada da preferência do serviço (os custos), e a decisão combina as duas de forma explícita.

### 2.5 Os custos esperados em função de p

Para ver todas as probabilidades de uma vez, desenhamos os quatro custos esperados de p = 0 a p = 1, para um modo e uma faixa da nota final. As faixas coloridas ao fundo mostram a ação escolhida em cada valor de p. O eixo vertical é cortado acima do custo da revisão manual, para que os custos pequenos fiquem visíveis; as retas de aceitar e rejeitar continuam subindo além do topo.

In [ ]:
CORES_ACOES = {"MATCH": "#55A868", "NONMATCH": "#C44E52", "DEFERRED": "#4C72B0", "LLM_REVIEW": "#DD8452"}
NOMES_ACOES = {"MATCH": "aceitar (MATCH)", "NONMATCH": "rejeitar (NONMATCH)",
               "DEFERRED": "revisão manual (DEFERRED)", "LLM_REVIEW": "consulta ao LLM (LLM_REVIEW)"}
GRADE_P = np.linspace(0, 1, 501)
# Faixas do motor dentro da faixa intermediária da nota final (de 5 a 9).
FAIXAS_INTERMEDIARIAS = [d.name for d in cfg.bands.definitions if d.min >= 5 and d.max <= 9]
NOMES_FAIXA = {d.name: f"nota final de {br(d.min, 0)} a {br(d.max, 0)}" for d in cfg.bands.definitions}


def janelas_de_acao(modo, faixa):
    # Intervalos de p, na grade de 0 a 1, em que cada ação tem o menor custo.
    acao = custos_esperados_v31(GRADE_P, modo, faixa)["ação de menor custo"].to_numpy()
    mudancas = np.flatnonzero(acao[1:] != acao[:-1]) + 1
    inicios = np.r_[0, mudancas]
    fins = np.r_[mudancas, len(GRADE_P) - 1]
    return pd.DataFrame({"ação de menor custo": acao[inicios],
                         "de p =": GRADE_P[inicios], "até p =": GRADE_P[fins]})


def desenhar_custos(modo, faixa):
    tabela = custos_esperados_v31(GRADE_P, modo, faixa)
    fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
    for acao in ACOES_V31:
        ax.plot(GRADE_P, tabela[acao], color=CORES_ACOES[acao], label=NOMES_ACOES[acao])
    for acao, inicio, fim in janelas_de_acao(modo, faixa).itertuples(index=False):
        ax.axvspan(inicio, fim, color=CORES_ACOES[acao], alpha=0.12)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 3 * tabela["DEFERRED"].max())
    ax.set_title(f"Custo esperado por ação: {NOMES_MODO[modo]}, {NOMES_FAIXA[faixa]}")
    ax.set_xlabel("Probabilidade calibrada p de ser par verdadeiro")
    ax.set_ylabel("Custo esperado (revisões equivalentes)")
    ax.legend(fontsize="small", loc="center left", bbox_to_anchor=(1.01, 0.5))
    return fig


print("Funções janelas_de_acao e desenhar_custos prontas.")

**🎛️ Controle: modo e faixa.** A célula desenha primeiro a figura da vigilância na faixa de nota final de 6 a 7; essa figura fica na saída mesmo sem controles interativos e no arquivo exportado. Logo abaixo, os controles (quando o Jupyter tem `ipywidgets`) redesenham uma cópia para o modo e a faixa escolhidos, entre as quatro faixas da faixa intermediária da nota final. A tabela mostra os intervalos de p em que cada ação tem o menor custo.

In [ ]:
def _ui_custos(modo, faixa):
    fig = desenhar_custos(modo, faixa)
    plt.show()
    plt.close(fig)
    mostrar(janelas_de_acao(modo, faixa))


_ui_custos("vigilancia", "grey_mid")  # figura estática, sempre na saída
nbh.interagir(_ui_custos, {
    "modo": nbh.Escolha([(NOMES_MODO[m], m) for m in nf.CUSTOS_POR_MODO], "vigilancia", "Modo"),
    "faixa": nbh.Escolha([(NOMES_FAIXA[f], f) for f in FAIXAS_INTERMEDIARIAS], "grey_mid", "Faixa"),
})

**Duas faixas, duas leituras.** A célula imprime, para a vigilância, os intervalos de p em que cada ação vence nas faixas de nota final de 5 a 6 e de 6 a 7, junto com o que o revisor de referência da tese fez nessas faixas. Depois conta, nas quatro faixas intermediárias e nos dois modos, em que parte da escala de p cada ação vence.

In [ ]:
for faixa in ("grey_low", "grey_mid"):
    decisoes_ref, pares_ref, _, _, fn_ref, _ = nf.CONTAGENS_LLM_HAIKU_V4["vigilancia"][faixa]
    print(f"Vigilância, {NOMES_FAIXA[faixa]}: o revisor de referência rejeitou {inteiro(fn_ref)} "
          f"dos {inteiro(pares_ref)} pares verdadeiros que viu, em {inteiro(decisoes_ref)} decisões.")
    mostrar(janelas_de_acao("vigilancia", faixa))

fracoes_acao = pd.DataFrame({
    (NOMES_MODO[modo], NOMES_FAIXA[faixa]): custos_esperados_v31(GRADE_P, modo, faixa)["ação de menor custo"]
    .value_counts(normalize=True).reindex(ACOES_V31, fill_value=0.0)
    for modo in nf.CUSTOS_POR_MODO
    for faixa in FAIXAS_INTERMEDIARIAS
}).T
print("Parte da escala de p em que cada ação tem o menor custo:")
display(fracoes_acao.map(pct))

fracoes_vig = fracoes_acao.loc[NOMES_MODO["vigilancia"]]
faixa_baixa, faixa_media = NOMES_FAIXA["grey_low"], NOMES_FAIXA["grey_mid"]
_, pares_baixa, _, _, fn_baixa, _ = nf.CONTAGENS_LLM_HAIKU_V4["vigilancia"]["grey_low"]
confere("na vigilância, revisar (manualmente ou com o LLM) vence em mais da metade da escala de p "
        "nas quatro faixas intermediárias",
        (fracoes_vig[["DEFERRED", "LLM_REVIEW"]].sum(axis=1) > 0.5).all())
confere(f"na vigilância, com {faixa_media}, o LLM substitui a revisão manual em toda a escala de p",
        fracoes_vig.loc[faixa_media, "DEFERRED"] == 0 and fracoes_vig.loc[faixa_media, "LLM_REVIEW"] > 0)
confere(f"na vigilância, com {faixa_baixa}, o revisor de referência rejeitou todos os pares verdadeiros "
        "que viu, a consulta ao LLM nunca compensa e a revisão manual vence em mais da metade da escala",
        fn_baixa == pares_baixa and fracoes_vig.loc[faixa_baixa, "LLM_REVIEW"] == 0
        and fracoes_vig.loc[faixa_baixa, "DEFERRED"] > 0.5)

**Leitura.** Os custos de aceitar e de rejeitar são retas que se cruzam num valor de p que depende só da razão entre os custos dos erros. O custo da revisão manual muda pouco com p e fica abaixo das duas retas numa janela larga: ali a política manda revisar, e não só no meio da escala. O custo do LLM depende das taxas da faixa. Na faixa de nota final de 6 a 7, em que o LLM errou pouco, ele substitui a revisão manual em toda a escala, porque a consulta custa uma fração de uma revisão. Na faixa de 5 a 6, em que o revisor de referência rejeitou todos os pares verdadeiros que viu, o custo do LLM sobe com p quase como a reta de rejeitar: a consulta nunca compensa, e a revisão manual vence numa janela larga. As frases conferidas acima garantem que essa leitura vale nesta execução.

## 3. Dados sintéticos: uma base que imita a dificuldade da tese

A função `gerar_base`, do módulo `synthetic_linkage`, sorteia óbitos e notificações e forma os potenciais pares em três passos de blocagem (6, 9 e 11). Cada potencial par traz os 29 subescores do CdR, a nota final, o escore do OpenRecLink e o ponto de corte do seu passo, com os mesmos nomes de colunas do extrato usado na tese. Nenhum nome, endereço ou data é gerado: só códigos e números.

O que a base simula, e por quê:

- **Registros precários.** Em parte dos potenciais pares que são da mesma pessoa, os registros são precários: nome escrito de outro jeito, nascimento ausente ou aproximado, sem nome da mãe e sem endereço. São eles que tendem a cair abaixo do corte.
- **Duas maneiras de comparar os mesmos registros.** O OpenRecLink compara o nome pela pronúncia (código fonético) e tolera data de nascimento próxima ou ausente. O CdR compara fragmentos exatos do nome e padrões fixos de erro na data, e registra campo ausente como zero, igual à discordância. Por isso o escore e a nota final discordam justamente nos registros precários.
- **Não pares difíceis.** Homônimos (mesmo nome, outra mãe e outro nascimento) e familiares (mesma mãe e endereço, outro primeiro nome) parecem a mesma pessoa em parte dos campos.
- **Passo 11 fraco.** A chave do passo 11 (código fonético do último nome e sexo) junta muitos registros: esse passo concentra a maior parte dos potenciais pares e dos registros precários.
- **Viés de verificação.** Como na tese, todo potencial par acima do ponto de corte recebe PAR = 1, e abaixo dele só a revisão manual de origem cria PAR = 2. Na simulação, essa revisão encontra a mesma pessoa com probabilidade maior quanto mais perto do corte o potencial par estiver.

> **Atenção.** Os parâmetros do gerador foram escolhidos para a base se parecer com a **dificuldade** do problema da tese (pares verdadeiros raros abaixo do corte, concentração no passo 11, escore e nota final discordantes), não com os seus números. Os resultados deste notebook demonstram o método; não são evidência sobre ele.

As colunas `par_verdadeiro`, `registro_precario` e `tipo_par` guardam a verdade da simulação, que a tese não tem. Neste texto, essa verdade se chama **mesma pessoa (verdade da simulação)**, para não confundir com o par verdadeiro, que é a classe da referência operacional. Ela serve só para explicar o mecanismo (fim desta seção) e nunca entra como atributo.

**Gerar a base.** A célula gera a base com a semente fixa, conta potenciais pares, óbitos e notificações e mostra cinco potenciais pares (um com PAR = 1, um com PAR = 2 e três com PAR = 0) com algumas colunas: identificadores, passo, três subescores, a nota final, o escore do OpenRecLink, o ponto de corte e a referência operacional.

In [ ]:
t0 = time.perf_counter()
base = sl.gerar_base(seed=SEED)
print(f"Base gerada em {br(time.perf_counter() - t0, 1)} s.")
print(f"Potenciais pares.......: {inteiro(len(base))}")
print(f"Óbitos (REFREC)........: {inteiro(base['REFREC'].nunique())}")
print(f"Notificações (COMPREC).: {inteiro(base['COMPREC'].nunique())}")
colunas_vista = ["REFREC", "COMPREC", "PASSO", "NOME qtd frag iguais", "NOMEMAE qtd frag iguais",
                 "DTNASC dt iguais", "nota final", "escore_orl", "ponto_de_corte", "acima_corte", "PAR"]
amostra = pd.concat([
    base.loc[base["PAR"].eq(1)].head(1),
    base.loc[base["PAR"].eq(2)].head(1),
    base.loc[base["PAR"].eq(0)].head(3),
])
mostrar(amostra[colunas_vista])

**Contagens por passo.** A tabela separa, em cada passo, os potenciais pares acima e abaixo do ponto de corte e conta os PAR = 1 e PAR = 2. A partir daqui, `abaixo` designa os potenciais pares da *grey area*: é sobre eles que a fila de revisão manual trabalha.

In [ ]:
abaixo = base.loc[~base["acima_corte"]]
por_passo = pd.DataFrame({
    "potenciais pares": base.groupby("PASSO").size(),
    "acima do corte": base["acima_corte"].groupby(base["PASSO"]).sum(),
    "abaixo do corte": (~base["acima_corte"]).groupby(base["PASSO"]).sum(),
    "PAR = 1": base["PAR"].eq(1).groupby(base["PASSO"]).sum(),
    "PAR = 2": base["PAR"].eq(2).groupby(base["PASSO"]).sum(),
})
por_passo.loc["total"] = por_passo.sum()
cortes = base.groupby("PASSO")["ponto_de_corte"].first()
print("Ponto de corte: " + "; ".join(f"passo {int(passo)}, {br(corte, 1)}" for passo, corte in cortes.items()))
mostrar(por_passo)

n_acima, n_par2 = int(base["acima_corte"].sum()), int(abaixo["PAR"].eq(2).sum())
print(f"Acima do corte: {inteiro(n_acima)} potenciais pares ({pct(n_acima / len(base))} da base).")
print(f"Abaixo do corte: {inteiro(len(abaixo))} potenciais pares, dos quais {inteiro(n_par2)} "
      f"PAR = 2 ({pct(n_par2 / len(abaixo), 2)}).")
confere("todo potencial par acima do ponto de corte tem PAR = 1",
        base.loc[base["acima_corte"], "PAR"].eq(1).all())
confere("quase toda a base (mais de 90%) fica abaixo do corte", len(abaixo) > 0.9 * len(base))
confere("o passo 11 concentra a maior parte dos potenciais pares",
        por_passo.loc[11, "potenciais pares"] > 0.5 * len(base))
confere("abaixo do corte, os PAR = 2 são menos de 1% dos potenciais pares", n_par2 / len(abaixo) < 0.01)

**Leitura.** O corte aceita poucos potenciais pares, e todos viram PAR = 1, como na tese. Quase toda a base fica abaixo dele, e lá os PAR = 2 são raríssimos: revisar a *grey area* inteira para achá-los seria um esforço desproporcional. É esse desequilíbrio que torna a ordem da fila decisiva.

**O escore e a nota final separam os PAR = 2?** Abaixo do corte, comparamos a distribuição de dois valores entre PAR = 0 e PAR = 2: a distância do escore ao ponto de corte do passo (negativa, porque estamos abaixo dele) e a nota final. Como os grupos têm tamanhos muito diferentes, cada histograma é normalizado para área 1 (densidade). Depois, a célula conta quantos PAR = 0 e PAR = 2 há acima da mediana dos PAR = 2, em cada valor.

In [ ]:
distancia = abaixo["escore_orl"] - abaixo["ponto_de_corte"]
par2 = abaixo["PAR"].eq(2)
grupos_par = [("PAR = 0 (não par segundo a referência)", ~par2, "#4C72B0"),
              ("PAR = 2 (par verdadeiro segundo a referência)", par2, "#C44E52")]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for nome, mascara, cor in grupos_par:
    ax1.hist(distancia[mascara], bins=40, density=True, alpha=0.6, color=cor, label=nome)
    ax2.hist(abaixo.loc[mascara, "nota final"], bins=40, range=(0, 10), density=True,
             alpha=0.6, color=cor, label=nome)
ax1.set_title("Escore do OpenRecLink abaixo do corte")
ax1.set_xlabel("Escore menos ponto de corte do passo")
ax1.set_ylabel("Densidade (área total = 1)")
ax2.set_title("Nota final do CdR abaixo do corte")
ax2.set_xlabel("Nota final (0 a 10)")
ax2.set_ylabel("Densidade (área total = 1)")
for ax in (ax1, ax2):
    ax.legend(fontsize="small")
fig.tight_layout()
plt.show()

sobreposicao = {}
for nome_valor, valores in [("distância do escore ao corte", distancia), ("nota final", abaixo["nota final"])]:
    mediana_par2 = valores[par2].median()
    n2, n0 = int((valores[par2] >= mediana_par2).sum()), int((valores[~par2] >= mediana_par2).sum())
    sobreposicao[nome_valor] = (n2, n0)
    print(f"Com {nome_valor} igual ou acima da mediana dos PAR = 2: "
          f"{inteiro(n2)} PAR = 2 e {inteiro(n0)} PAR = 0.")
confere("os PAR = 2 ficam, em mediana, mais perto do corte que os PAR = 0",
        distancia[par2].median() > distancia[~par2].median())
confere("os PAR = 2 têm, em mediana, nota final maior que os PAR = 0",
        abaixo.loc[par2, "nota final"].median() > abaixo.loc[~par2, "nota final"].median())
confere("acima da mediana dos PAR = 2, os PAR = 0 são mais numerosos que os PAR = 2, nos dois valores",
        all(n0 > n2 for n2, n0 in sobreposicao.values()))

**Leitura.** Cada histograma foi normalizado para ter área 1: a figura compara formas, não quantidades. Os PAR = 2 se concentram perto do corte e nas notas finais altas, mas os PAR = 0 são muito mais numerosos; mesmo na metade de cima dos PAR = 2, em cada valor, os PAR = 0 são a maioria (contagens impressas acima). É essa sobreposição no topo de qualquer fila ordenada só pelo escore ou só pela nota final que motiva combinar os 29 subescores e a nota final num classificador (seção 4).

**O que a referência não vê.** Só nesta simulação sabemos quais potenciais pares são da mesma pessoa (verdade da simulação). A célula compara, abaixo do corte, as mesmas pessoas que a revisão manual de origem simulada encontrou (PAR = 2) com as que ela não encontrou e ficaram com PAR = 0. Na tese, essas mesmas pessoas não encontradas são invisíveis: contam como não pares na avaliação.

In [ ]:
mesmas_pessoas = abaixo.loc[abaixo["par_verdadeiro"].eq(1)]
passo_m = mesmas_pessoas["PASSO"]
vies = pd.DataFrame({
    "mesma pessoa (verdade da simulação)": passo_m.value_counts().sort_index(),
    "encontradas pela revisão de origem (PAR = 2)": mesmas_pessoas["PAR"].eq(2).groupby(passo_m).sum(),
    "de registro precário": mesmas_pessoas["registro_precario"].groupby(passo_m).sum(),
})
vies.loc["total"] = vies.sum()
mostrar(vies)
distancia_m = mesmas_pessoas["escore_orl"] - mesmas_pessoas["ponto_de_corte"]
encontrada = mesmas_pessoas["PAR"].eq(2)
print(f"Distância mediana ao corte: encontradas {br(distancia_m[encontrada].median(), 1)}; "
      f"não encontradas {br(distancia_m[~encontrada].median(), 1)}.")
confere("a revisão manual de origem simulada encontra sobretudo as mesmas pessoas mais perto do corte",
        distancia_m[encontrada].median() > distancia_m[~encontrada].median())

**Leitura.** Isso é o **viés de verificação**: a referência operacional favorece as mesmas pessoas de escore alto e, com isso, favorece na avaliação as filas que usam o escore (O1 e F4). Na tese há também o risco de **viés de incorporação**: se a revisora de origem viu as evidências do CdR, a referência favoreceria a ordenação pela nota final (O2) e pela probabilidade calibrada (O3). Sem suposições sobre o mecanismo de verificação, a direção dominante e a magnitude do viés não são identificáveis (tese, seção `sec:concl-limitacoes`). Aqui só o viés de verificação foi embutido, de propósito. Daqui em diante, todas as avaliações usam só PAR, como na tese.

## 4. Probabilidade calibrada fora da amostra

O escore do OpenRecLink e a nota final resumem a comparação em um número cada, com regras fixas. Na tese, as filas recomendadas partem de um classificador que combina os 29 subescores e a nota final, 30 atributos ao todo. O classificador é o **XGBoost** (muitas árvores de decisão pequenas, cada uma corrigindo os erros das anteriores), com os hiperparâmetros da tese. O alvo é a referência operacional: PAR igual a 1 ou 2 conta como par verdadeiro. Como há muito mais não pares, o ajuste usa pesos de classe que compensam o desbalanceamento. O escore do OpenRecLink, o passo e os identificadores ficam fora dos atributos.

Três cuidados tornam a probabilidade utilizável:

1. **Fora da amostra.** Cada potencial par recebe a probabilidade de um modelo que não o viu no ajuste; senão, o modelo "lembra" a resposta e parece melhor do que é.
2. **Agrupada por óbito.** Todos os potenciais pares de um mesmo óbito (`REFREC`) ficam do mesmo lado de cada partição: o modelo nunca é avaliado num óbito que já viu.
3. **Calibrada.** A probabilidade bruta do XGBoost passa por uma **calibração isotônica**: uma escada crescente, ajustada para que, em cada degrau, p acompanhe a fração observada de pares verdadeiros. O calibrador também é ajustado fora da amostra.

Para a fila, só a ordem importa: qualquer transformação crescente de p daria a mesma fila, a menos dos empates. A calibração serve para outra coisa: ler p como probabilidade (quantos em cada 100) e alimentar a política da seção 2, que multiplica p pelos custos dos erros.

Juntos, os três cuidados dão uma **validação cruzada aninhada 5 × 5**:

```text
base inteira, dividida por óbito em 5 partes
│
├── partição externa 1: teste = 1 parte; treino = as outras 4
│     ├── 5 partições internas dentro do treino
│     │     → probabilidades brutas fora da amostra para todo o treino
│     ├── calibrador isotônico ajustado nessas probabilidades
│     └── XGBoost reajustado no treino inteiro → pontua o teste → calibrador
├── partições externas 2 a 5: o mesmo, trocando a parte de teste
│
└── resultado: cada potencial par tem p de um modelo que nunca viu o seu óbito
```

São 30 ajustes do XGBoost: 25 internos e 5 finais. O modelo é ajustado na base inteira (acima e abaixo do corte) e avaliado abaixo do corte, onde a fila trabalha.

**As partições, antes de ajustar.** A função `particoes_aninhadas` monta as partições sem treinar nada. A tabela confere, em cada partição externa, os tamanhos do treino e do teste e quantos óbitos aparecem dos dois lados, que deve ser zero.

In [ ]:
assert "base" in globals(), (
    "A sessão perdeu as variáveis das seções anteriores (foi reiniciada?). "
    "Clique nesta célula e use Ambiente de execução > Executar antes (Ctrl+F8)."
)
y_base = nf.rotulo(base)
grupos_base = base["REFREC"].to_numpy()
particoes = nf.particoes_aninhadas(y_base, grupos_base, seed=SEED)
resumo_particoes = pd.DataFrame([
    {
        "partição externa": parte.fold + 1,
        "potenciais pares no treino": len(parte.treino),
        "potenciais pares no teste": len(parte.teste),
        "pares verdadeiros no teste": int(y_base[parte.teste].sum()),
        "óbitos nos dois lados": len(np.intersect1d(grupos_base[parte.treino], grupos_base[parte.teste])),
        "partições internas": len(parte.internas),
    }
    for parte in particoes
]).set_index("partição externa")
mostrar(resumo_particoes)
confere("nenhum óbito aparece no treino e no teste da mesma partição externa",
        resumo_particoes["óbitos nos dois lados"].eq(0).all())

**Ajustar.** A função `prob_calibrada_oof` faz a validação cruzada aninhada inteira e devolve a probabilidade calibrada fora da amostra de cada potencial par. Ela recebe a base sem as colunas latentes, como a tese a conheceria. Esta é a célula mais demorada do notebook.

In [ ]:
t0 = time.perf_counter()
p_oof = nf.prob_calibrada_oof(sl.sem_colunas_latentes(base), seed=SEED)
tempo_oof = time.perf_counter() - t0
n_ajustes = len(particoes) * (len(particoes[0].internas) + 1)
print(f"Probabilidade calibrada fora da amostra para {inteiro(len(p_oof))} potenciais pares, "
      f"com {n_ajustes} ajustes do XGBoost, em {br(tempo_oof, 1)} s.")

**A probabilidade é calibrada?** Agrupamos os potenciais pares por valor de p (de 0 a 10%, de 10% a 20%, e assim por diante) e comparamos, em cada grupo, a média de p com a fração observada de pares verdadeiros. Se a probabilidade for calibrada, os pontos ficam perto da diagonal. O tamanho do ponto cresce com o número de potenciais pares no grupo. São duas figuras: a base inteira, onde o modelo foi ajustado, e só os potenciais pares abaixo do corte, onde a fila trabalha (ali, par verdadeiro quer dizer PAR = 2).

In [ ]:
y_abaixo = nf.rotulo(abaixo)
p_abaixo = p_oof.loc[abaixo.index]
paineis = {
    "Base inteira": (y_base, p_oof.to_numpy()),
    "Abaixo do corte": (y_abaixo, p_abaixo.to_numpy()),
}
fig, eixos = plt.subplots(1, 2, figsize=(11, 5))
pontos_por_painel = {}
for ax, (titulo, (y, p)) in zip(eixos, paineis.items(), strict=True):
    pontos = nbh.reliability_points(y, p, n_bins=10)
    pontos_por_painel[titulo] = pontos
    ax.plot([0, 1], [0, 1], linestyle="--", color="0.5", label="calibração perfeita")
    ax.scatter(pontos["conf"], pontos["acc"], s=30 + 300 * np.sqrt(pontos["n"] / pontos["n"].max()),
               color="#C44E52", label="grupos de p")
    ax.set_title(f"{titulo}: probabilidade calibrada fora da amostra")
    ax.set_xlabel("Probabilidade média prevista no grupo")
    ax.set_ylabel("Fração observada de pares verdadeiros")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc="upper left")
fig.tight_layout()
plt.show()

for titulo, pontos in pontos_por_painel.items():
    print(f"{titulo}:")
    mostrar(pontos.drop(columns="bin").rename(columns={
        "lo": "p de", "hi": "p até", "n": "potenciais pares", "conf": "p média",
        "acc": "fração observada", "gap": "diferença"}))
    escassos = pontos.loc[pontos["n"] < 30]
    if len(escassos):
        faixas_p = "; ".join(f"de {pct(lo, 0)} a {pct(hi, 0)}" for lo, hi in zip(escassos["lo"], escassos["hi"], strict=True))
        print(f"   Grupos com menos de 30 potenciais pares (fração observada instável): {faixas_p}.")
    vazios = 10 - len(pontos)
    if vazios:
        print(f"   Grupos de p sem nenhum potencial par: {vazios}.")
    grandes = pontos.loc[pontos["n"] >= 30]
    print(f"   Maior diferença entre p média e fração observada, nos grupos com 30 ou mais "
          f"potenciais pares: {br(grandes['gap'].max(), 3)}.")
no_primeiro_grupo = int((p_oof < 0.1).sum())
print(f"{pct(no_primeiro_grupo / len(base))} dos potenciais pares da base têm p abaixo de 10%.")
confere("mais de 80% dos potenciais pares têm p abaixo de 10%", no_primeiro_grupo > 0.8 * len(base))

**Leitura.** A grande maioria dos potenciais pares fica no primeiro grupo, com p muito pequena: é o retrato de uma base dominada por não pares. Os grupos listados como escassos têm fração observada instável, e a distância deles à diagonal não deve ser lida como erro de calibração. Compare os dois painéis: abaixo do corte, onde os pares verdadeiros são mais raros, os grupos de p maior ficam ainda mais escassos. A calibração é ajustada fora da amostra e, por isso, não é perfeita; o que importa é que p acompanhe a frequência observada onde há potenciais pares suficientes para medi-la.

**Os degraus da calibração isotônica.** A escada isotônica dá o mesmo valor de p a todos os potenciais pares de um degrau. Abaixo do corte, isso cria empates grandes, que a fila desfaz por sorteio com semente fixa. A célula mede esses empates; eles voltam na seção 6.

In [ ]:
degraus = p_abaixo.value_counts()
p_do_maior_degrau = degraus.idxmax()
no_maior_degrau = p_abaixo.eq(p_do_maior_degrau)
print(f"Valores distintos de p abaixo do corte: {inteiro(len(degraus))}.")
print(f"Maior empate: {inteiro(degraus.max())} potenciais pares com p = {br(p_do_maior_degrau, 5)}, "
      f"dos quais {inteiro(abaixo.loc[no_maior_degrau, 'PAR'].eq(2).sum())} são PAR = 2.")

**Por que partir por óbito, e não por potencial par.** Em dados reais, os potenciais pares de um mesmo óbito repetem o nome, a data de nascimento e o nome da mãe do falecido. Se uns caem no treino e outros no teste, o modelo pode reconhecer no teste um óbito que já viu no ajuste, e a avaliação fica otimista: mede o desempenho em óbitos parcialmente conhecidos, não em óbitos novos. Isso se chama **vazamento**. A tese usou a partição por óbito em todas as estimativas, e este notebook faz o mesmo.

Nesta base sintética, os subescores são sorteados linha a linha, sem nada em comum entre os potenciais pares de um mesmo óbito além do identificador; ela não tem, portanto, por onde vazar, e não serve para medir esse otimismo. Por isso o notebook não compara as duas partições. A função `prob_por_linha`, de `nb_fila`, faz a versão por potencial par, para quem quiser repetir a comparação com dados reais.

**Quem separa melhor os PAR = 2 abaixo do corte?** Comparamos três valores como ordenadores da *grey area*: o escore do OpenRecLink, a nota final do CdR e a probabilidade calibrada fora da amostra. Duas medidas:

- **AUC**: probabilidade de um PAR = 2 sorteado ficar acima de um potencial par sorteado entre os demais;
- **precisão média (AP)**: percorra a fila do topo; a cada PAR = 2 encontrado, anote a proporção de PAR = 2 entre os potenciais pares revisados até ali; a AP é a média dessas anotações. Ela resume a curva precisão-sensibilidade e pesa mais o topo da fila. Uma fila sorteada tem AP igual à proporção de PAR = 2.

Com pares verdadeiros raros, a AP é a que mais importa para a revisão manual, porque o revisor humano começa pelo topo.

In [ ]:
ordenadores = {
    "escore do OpenRecLink": abaixo["escore_orl"].to_numpy(),
    "nota final do CdR": abaixo["nota final"].to_numpy(),
    "probabilidade calibrada fora da amostra": p_abaixo.to_numpy(),
}
desempenho = pd.DataFrame({
    nome: {"AUC": roc_auc_score(y_abaixo, valores),
           "precisão média (AP)": average_precision_score(y_abaixo, valores)}
    for nome, valores in ordenadores.items()
}).T
print(f"AP de uma fila sorteada (proporção de PAR = 2 abaixo do corte): {br(y_abaixo.mean(), 4)}")
mostrar(desempenho)

ap, auc = desempenho["precisão média (AP)"], desempenho["AUC"]
confere("todas as AUC ficam acima de 0,8: a maioria dos não pares é fácil de separar", (auc > 0.8).all())
confere("a probabilidade calibrada tem precisão média maior que a nota final e que o escore",
        ap["probabilidade calibrada fora da amostra"] > max(ap["nota final do CdR"], ap["escore do OpenRecLink"]))
if auc["escore do OpenRecLink"] >= auc["probabilidade calibrada fora da amostra"]:
    print("Leitura desta execução: a AUC do escore é igual ou maior que a da probabilidade calibrada, "
          "como na tese; a vantagem da probabilidade está no topo da fila, que a AUC quase não vê.")
else:
    print("Leitura desta execução: a probabilidade calibrada tem AUC e precisão média maiores que o escore.")

**Leitura.** A AUC é alta para todos os ordenadores, porque a *grey area* está cheia de não pares fáceis; por isso ela diz pouco sobre o topo da fila. A precisão média mostra a diferença que interessa ao serviço: a probabilidade calibrada põe mais PAR = 2 no topo que a nota final e o escore (conferido acima). A seção 5 usa essa probabilidade para montar as filas.

## 5. Ordenações da fila abaixo do corte

Os vínculos aceitos pelo ponto de corte ficam como estão. A fila de revisão manual contém só os potenciais pares da *grey area*, em quatro ordens possíveis:

| Fila | Ordena por | Ideia |
|---|---|---|
| O1 | escore do OpenRecLink, do maior para o menor | continuar a lógica do próprio corte |
| O2 | nota final do CdR | usar a comparação detalhada do CdR |
| O3 | probabilidade calibrada fora da amostra (seção 4) | usar o classificador |
| F4 | fusão de O3 e O1 por posição recíproca (RRF) | combinar o classificador e o escore |

> **Nota sobre F4.** Na tese, F4 foi escolhida entre seis variantes de fusão avaliadas nos mesmos pares verdadeiros abaixo do corte, sem correção para multiplicidade; a regra foi fixada antes do cálculo, mas isso não equivale a pré-registro (seção 10).

### 5.1 A fusão por posição recíproca, num exemplo de brinquedo

A RRF não mistura os valores, porque o escore e a probabilidade têm escalas diferentes; mistura as **posições**. Cada item ganha, em cada fila, $1/(k + \text{posição})$, e a fusão ordena pela soma. Com $k = 60$, como na tese e no pacote, estar em primeiro vale $1/61$ e em segundo, $1/62$: as diferenças entre vizinhos são pequenas, e nenhuma fila sozinha manda na fusão.

A célula usa cinco itens, de A a E, com posições inventadas em O3 e O1, calcula a soma à mão e confere a ordem com a função `rrf` do pacote.

In [ ]:
assert "base" in globals(), (
    "A sessão perdeu as variáveis das seções anteriores (foi reiniciada?). "
    "Clique nesta célula e use Ambiente de execução > Executar antes (Ctrl+F8)."
)
from gzcmd_record_linkage.ranking import rrf

ITENS = ["A", "B", "C", "D", "E"]
FILA_O3_BRINQUEDO = ["A", "B", "C", "D", "E"]
FILA_O1_BRINQUEDO = ["B", "C", "D", "A", "E"]
K_RRF = 60


def para_indices(fila):
    # Converte uma fila de letras na lista de posições (A = 0, B = 1, ...) que o rrf recebe.
    return np.array([ITENS.index(item) for item in fila])


def posicoes(fila_indices):
    # Posição (1 = primeiro) de cada item, na ordem A, B, C, ...
    posicao = np.empty(len(fila_indices), dtype=int)
    posicao[fila_indices] = np.arange(1, len(fila_indices) + 1)
    return posicao


o3_brinquedo, o1_brinquedo = para_indices(FILA_O3_BRINQUEDO), para_indices(FILA_O1_BRINQUEDO)
soma_rrf = 1 / (K_RRF + posicoes(o3_brinquedo)) + 1 / (K_RRF + posicoes(o1_brinquedo))
fusao = rrf([o3_brinquedo, o1_brinquedo], k=K_RRF, seed=SEED, input_kind="orders")
assert (np.argsort(-soma_rrf, kind="stable") == fusao).all(), "a conta à mão difere do rrf do pacote"
tabela_rrf = pd.DataFrame({
    "posição em O3": posicoes(o3_brinquedo),
    "posição em O1": posicoes(o1_brinquedo),
    "soma de 1/(k + posição)": soma_rrf,
    "posição na fusão": posicoes(fusao),
}, index=ITENS).sort_values("posição na fusão")
mostrar(tabela_rrf, casas=5)
print("Fila O3 .: " + ", ".join(FILA_O3_BRINQUEDO))
print("Fila O1 .: " + ", ".join(FILA_O1_BRINQUEDO))
print("Fusão F4 : " + ", ".join(ITENS[i] for i in fusao))
print("✓ a conta à mão dá a mesma ordem que o rrf do pacote.")
pos_fusao = dict(zip(ITENS, posicoes(fusao), strict=True))
print(f"A, primeiro em O3 e quarto em O1, termina na posição {pos_fusao['A']} da fusão; "
      f"B, segundo em O3 e primeiro em O1, na posição {pos_fusao['B']}.")

**Leitura.** A fusão premia quem aparece bem nas duas filas. Um item excelente numa fila e mediano na outra perde algumas posições para quem está bem colocado nas duas, mas não despenca. É assim que F4 usa o escore do OpenRecLink para desempatar e corrigir a probabilidade calibrada, e vice-versa.

### 5.2 As quatro filas na *grey area*

A função `ordenacoes` monta as quatro filas sobre os potenciais pares abaixo do corte. Os empates, comuns na nota final e nos degraus da probabilidade, são desfeitos por sorteio com semente fixa, nunca pela ordem das linhas nem pela classe. A célula conta quantos PAR = 2 cada fila põe entre os primeiros 10, 50, 100 e 500 potenciais pares e diz qual fila lidera com 10 e com 100 revisões.

In [ ]:
ordens = nf.ordenacoes(abaixo, p_oof, seed=SEED)
topo = pd.DataFrame({
    nome: {f"primeiros {inteiro(n)}": int(y_abaixo[ordem[:n]].sum()) for n in (10, 50, 100, 500)}
    for nome, ordem in ordens.items()
})
print(f"PAR = 2 abaixo do corte: {inteiro(y_abaixo.sum())}. Uma fila sorteada poria, em média, "
      f"{br(100 * y_abaixo.mean(), 2)} entre os primeiros 100.")
mostrar(topo)
for n in (10, 100):
    linha = topo.loc[f"primeiros {inteiro(n)}"]
    lideres = linha.index[linha == linha.max()].tolist()
    empate = " (empate)" if len(lideres) > 1 else ""
    print(f"Com {n} revisões, lidera {', '.join(lideres)}{empate}, com {inteiro(linha.max())} PAR = 2.")
confere("as quatro filas põem entre os primeiros 100 mais PAR = 2 que uma fila sorteada",
        (topo.loc["primeiros 100"] > 100 * y_abaixo.mean()).all())

**Leitura.** As quatro filas põem no topo mais PAR = 2 que o sorteio (conferido acima), mas não igualmente: veja nas linhas impressas qual fila lidera com 10 e com 100 revisões, e se a liderança muda entre essas cargas. O topo da fila é onde cabe o orçamento humano típico; a seção 6 olha a fila inteira.

## 6. Curva de esforço: quantas revisões para cada meta

A **curva de esforço** de uma fila conta quantos PAR = 2 o revisor humano recupera depois de cada revisão, seguindo a fila do começo. Supomos um **revisor perfeito simulado**: todo par verdadeiro revisado é recuperado, e nenhum não par vira vínculo.

As metas são frações dos PAR = 2 da base: 25%, 64% e 100%, as mesmas frações das metas da tese (seção 10). A célula responde a duas perguntas que são a mesma curva lida de dois jeitos: quantas revisões cada fila exige para cada meta e quantos PAR = 2 cada fila recupera com a mesma carga de revisões. Quando duas filas empatam no menor número de revisões, a célula diz.

In [ ]:
assert "base" in globals(), (
    "A sessão perdeu as variáveis das seções anteriores (foi reiniciada?). "
    "Clique nesta célula e use Ambiente de execução > Executar antes (Ctrl+F8)."
)
curvas = {nome: nf.curva_esforco(ordem, y_abaixo) for nome, ordem in ordens.items()}
total_par2 = int(y_abaixo.sum())
FRACOES_META = {"25%": 0.25, "64%": 0.64, "100%": 1.0}
metas = {rotulo_meta: max(1, int(np.floor(fracao * total_par2 + 0.5)))
         for rotulo_meta, fracao in FRACOES_META.items()}
revisoes = pd.DataFrame({
    f"meta de {rotulo_meta} ({meta} PAR = 2)": {
        nome: nf.revisoes_para_meta(curva, meta) for nome, curva in curvas.items()
    }
    for rotulo_meta, meta in metas.items()
})
print("Revisões manuais necessárias para cada meta:")
mostrar(revisoes)
for coluna in revisoes:
    minimo = revisoes[coluna].min()
    vencedoras = revisoes.index[revisoes[coluna] == minimo].tolist()
    empate = " (empate)" if len(vencedoras) > 1 else ""
    print(f"{coluna}: menos revisões com {', '.join(vencedoras)}{empate}: {inteiro(minimo)}.")

CARGAS = (20, 50, 100, 500, 1000)
mesma_carga_filas = pd.DataFrame({
    f"{inteiro(carga)} revisões": {nome: nf.recuperados_com_carga(curva, carga) for nome, curva in curvas.items()}
    for carga in CARGAS
})
print("PAR = 2 recuperados com a mesma carga de revisões:")
mostrar(mesma_carga_filas)

**A figura.** O eixo horizontal está em escala logarítmica, para mostrar na mesma figura as primeiras dezenas de revisões e a fila inteira. As linhas pontilhadas marcam as metas, com o rótulo na borda direita.

In [ ]:
CORES_FILAS = {"O1": "#4C72B0", "O2": "#8172B3", "O3": "#C44E52", "F4": "#55A868"}


def desenhar_curvas(curvas, orcamento=None):
    fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
    for nome, curva in curvas.items():
        ax.step(curva.index, curva.to_numpy(), where="post", color=CORES_FILAS[nome], label=nome)
    for rotulo_meta, meta in metas.items():
        ax.axhline(meta, color="0.6", linestyle=":", linewidth=1)
        ax.annotate(f"meta de {rotulo_meta} ({meta} PAR = 2)", xy=(0.99, meta),
                    xycoords=("axes fraction", "data"), xytext=(0, 3), textcoords="offset points",
                    ha="right", va="bottom", fontsize="small", color="0.3")
    if orcamento is not None:
        ax.axvline(orcamento, color="black", linestyle="--",
                   label=f"orçamento: {inteiro(orcamento)} revisões")
    ax.set_xscale("log")
    ax.set_title("Curva de esforço abaixo do ponto de corte")
    ax.set_xlabel("Revisões manuais feitas, seguindo a fila (escala logarítmica)")
    ax.set_ylabel("PAR = 2 recuperados (revisor perfeito simulado)")
    ax.legend(loc="upper left")
    return fig


fig = desenhar_curvas(curvas)
plt.show()
plt.close(fig)

dominantes = [fila for fila in revisoes.index if (revisoes.loc[fila] <= revisoes.min()).all()]
confere("nenhuma fila exige o menor número de revisões em todas as metas", not dominantes)
confere("O3 exige menos revisões que O1 nas metas de 25% e de 64%",
        (revisoes.loc["O3"].iloc[:2] < revisoes.loc["O1"].iloc[:2]).all())

**Leitura.** Leia a figura da esquerda para a direita, como o serviço leria: com poucas revisões, ganha a fila cuja curva sobe primeiro; com muitas, a que chega antes ao topo. Nenhuma fila exige menos revisões em todas as metas (conferido acima), e a melhor fila depende do orçamento humano. Por isso a recomendação não é uma fila fixa, e sim escolher entre O3 e F4 pela curva de esforço, dado o orçamento.

Na recuperação integral, O3 pode exigir mais revisões que O1, como aconteceu na tese. A célula seguinte compara as filas nessa meta, examina o último PAR = 2 da fila O3 e conta, pela verdade da simulação, quantos dos PAR = 2 que O3 só alcança depois da meta de 64% vêm de registro precário.

In [ ]:
p_abaixo_vetor = p_abaixo.to_numpy()
revisoes_o3_total = nf.revisoes_para_meta(curvas["O3"], total_par2)
ultimo_o3 = ordens["O3"][revisoes_o3_total - 1]
empatados = int((p_abaixo_vetor == p_abaixo_vetor[ultimo_o3]).sum())
mais_provaveis = int((p_abaixo_vetor > p_abaixo_vetor[ultimo_o3]).sum())
print(f"O último PAR = 2 da fila O3 está na posição {inteiro(revisoes_o3_total)}, com "
      f"p = {br(p_abaixo_vetor[ultimo_o3], 5)}. {inteiro(mais_provaveis)} potenciais pares têm p maior, "
      f"e {inteiro(empatados)} (contando ele) têm exatamente esse p: entre eles, a ordem vem do sorteio.")
n_64 = nf.revisoes_para_meta(curvas["O3"], metas["64%"])
tardios = abaixo.iloc[ordens["O3"][n_64:]]
tardios = tardios.loc[tardios["PAR"].eq(2)]
print(f"PAR = 2 que O3 só alcança depois da meta de 64%: {inteiro(len(tardios))}; destes, "
      f"{inteiro(tardios['registro_precario'].sum())} vêm de registro precário (verdade da simulação).")
r_o3, r_o1, r_f4 = (revisoes.loc[fila].iloc[-1] for fila in ("O3", "O1", "F4"))
if r_o3 > r_o1:
    print(f"Leitura desta execução: para recuperar todos, O3 exige mais revisões que O1 "
          f"({inteiro(r_o3)} contra {inteiro(r_o1)}); F4 exige {inteiro(r_f4)}.")
else:
    print(f"Leitura desta execução: para recuperar todos, O3 não exige mais revisões que O1 "
          f"({inteiro(r_o3)} contra {inteiro(r_o1)}); F4 exige {inteiro(r_f4)}.")

**Leitura.** Os últimos PAR = 2 de O3 têm probabilidade baixa; a saída mostra quantos deles vêm de registro precário, parecido com não pares em quase todos os subescores. Quando caem num degrau da escada isotônica compartilhado com muitos não pares, só o sorteio decide sua posição dentro do degrau. A fusão F4 usa também a posição no escore do OpenRecLink, que distingue esses potenciais pares de outro modo; por isso tende a ajudar no fim da fila.

Parte da vantagem de F4 e de O1 no fim da fila pode vir da própria referência: a revisão manual de origem encontrou sobretudo os pares de escore alto (viés de verificação, seção 3), o que favorece as filas que usam o escore. Na direção oposta, o viés de incorporação favoreceria O2 e O3. A direção dominante não é identificável (tese, seção `sec:concl-limitacoes`).

**🎛️ Controle: o orçamento humano.** Escolha quantas revisões manuais o serviço consegue fazer. A célula desenha primeiro a figura estática para 100 revisões, que fica na saída e no arquivo exportado; logo abaixo, o controle (quando o Jupyter tem `ipywidgets`) redesenha uma cópia para o orçamento escolhido. A tabela mostra quantos PAR = 2 cada fila recupera com essa carga.

In [ ]:
ORCAMENTO_PADRAO = 100


def _ui_orcamento(orcamento):
    fig = desenhar_curvas(curvas, orcamento)
    plt.show()
    plt.close(fig)
    recuperados = pd.Series(
        {nome: nf.recuperados_com_carga(curva, orcamento) for nome, curva in curvas.items()},
        name=f"PAR = 2 recuperados com {inteiro(orcamento)} revisões",
    )
    mostrar(recuperados)


_ui_orcamento(ORCAMENTO_PADRAO)  # figura estática, sempre na saída
nbh.interagir(_ui_orcamento, {
    "orcamento": nbh.Deslizante(ORCAMENTO_PADRAO, 10, 2000, 10, "Revisões", inteiro=True),
})

**Leitura.** Compare um orçamento pequeno, de algumas dezenas de revisões, com um grande, de centenas: a fila que mais recupera pode mudar (veja também a tabela de mesma carga, no começo desta seção). É essa leitura que o serviço faria, com uma amostra revisada da sua própria base, para escolher a fila e o ponto de parada.

## 7. Incerteza: o *bootstrap* agrupado por óbito

Uma diferença de revisões entre duas filas pode ser acaso: com outros óbitos na base, ela mudaria. O ***bootstrap* agrupado** imita essa variação com a própria base:

1. sorteia, com reposição, tantos óbitos quantos há abaixo do corte (alguns saem mais de uma vez, outros nenhuma);
2. monta a réplica com os potenciais pares desses óbitos, cada um repetido tantas vezes quantas o seu óbito saiu;
3. conta quantas revisões cada uma das duas filas exige para a meta, na mesma réplica, e guarda a diferença (primeira fila menos segunda);
4. repete 2.000 vezes.

**Réplicas válidas** são as que ficaram com ao menos um PAR = 2; as outras são descartadas. Quando uma réplica tem menos PAR = 2 que a base, a meta passa a ser a mesma fração dos PAR = 2 da réplica, como na tese.

O **IC 95%** é o intervalo entre os percentis 2,5% e 97,5% das diferenças. Valores negativos favorecem a primeira fila (menos revisões). Um **IC que exclui zero** tem as duas pontas do mesmo lado de zero: em pelo menos 97,5% das réplicas, a mesma fila exigiu menos revisões. Um IC que inclui zero quer dizer que o *bootstrap* não distingue a diferença do acaso desta base.

Usamos a meta de 64% e três comparações: O3 com O1, F4 com O1 e F4 com O3.

In [ ]:
assert "base" in globals(), (
    "A sessão perdeu as variáveis das seções anteriores (foi reiniciada?). "
    "Clique nesta célula e use Ambiente de execução > Executar antes (Ctrl+F8)."
)
META_BOOTSTRAP = metas["64%"]
N_REPLICAS = 2000
COMPARACOES_BOOTSTRAP = [("O3", "O1"), ("F4", "O1"), ("F4", "O3")]
coluna_meta = revisoes.columns[1]
t0 = time.perf_counter()
bootstrap = pd.DataFrame({
    f"{a} menos {b}": nf.bootstrap_diferenca(
        abaixo, ordens[a], ordens[b], meta=META_BOOTSTRAP, n=N_REPLICAS, seed=SEED
    )
    for a, b in COMPARACOES_BOOTSTRAP
}).T
print(f"Bootstrap agrupado, meta de {META_BOOTSTRAP} PAR = 2, {inteiro(N_REPLICAS)} réplicas por "
      f"comparação: {br(time.perf_counter() - t0, 1)} s.")


def _diferenca(valor):
    # Diferenças de revisões: inteiras sem casas; medianas fracionárias com uma casa.
    return inteiro(valor) if float(valor).is_integer() else br(valor, 1)


pontuais = [revisoes.loc[a, coluna_meta] - revisoes.loc[b, coluna_meta] for a, b in COMPARACOES_BOOTSTRAP]
display(pd.DataFrame({
    "réplicas válidas": [inteiro(v) for v in bootstrap["réplicas válidas"]],
    "diferença pontual": [_diferenca(v) for v in pontuais],
    "diferença mediana (primeira fila menos segunda)": [_diferenca(v) for v in bootstrap["diferença mediana (A menos B)"]],
    "IC 95% inferior": [_diferenca(v) for v in bootstrap["IC95 inferior"]],
    "IC 95% superior": [_diferenca(v) for v in bootstrap["IC95 superior"]],
    "proporção em que a primeira fila exige menos": [pct(v) for v in bootstrap["proporção em que A exige menos"]],
}, index=bootstrap.index))

for (a, b), (comparacao, linha) in zip(COMPARACOES_BOOTSTRAP, bootstrap.iterrows(), strict=True):
    inferior, superior = linha["IC95 inferior"], linha["IC95 superior"]
    if superior < 0:
        leitura = f"o IC exclui zero: em pelo menos 97,5% das réplicas, {a} exigiu menos revisões que {b}"
    elif inferior > 0:
        leitura = f"o IC exclui zero: em pelo menos 97,5% das réplicas, {b} exigiu menos revisões que {a}"
    else:
        leitura = "o IC inclui zero: o bootstrap não distingue a diferença do acaso"
    print(f"{comparacao}: {leitura}.")

**Leitura.** O IC é condicional: as probabilidades e as filas ficam fixas em todas as réplicas (o modelo não é reajustado, e neste notebook a fusão não é refeita), e a referência operacional é tomada como certa. Ele mede a variação devida a quais óbitos entraram na base, não a incerteza do treinamento nem a da referência. Com poucos PAR = 2, como aqui e na tese, os intervalos são largos, e um IC que exclui zero por pouco pede cautela. A última coluna da tabela mostra em que proporção das réplicas a primeira fila exigiu menos revisões que a segunda.

## 8. Contraste com a automação: a política de quatro ações com LLM

Até aqui, o revisor humano revisa a fila na ordem escolhida. A alternativa avaliada na tese é automatizar parte das decisões abaixo do corte com a política v3.1 da seção 2: guardas, custos esperados e consulta a um LLM. A função `politica_v31` aplica essa política à *grey area* com:

- a probabilidade calibrada da seção 4;
- os custos da tese em cada modo;
- um **LLM simulado**, que não é um modelo real: ele erra de propósito, com as taxas por faixa do revisor de referência da tese (Claude Haiku 5.5), e a classe vem da referência operacional;
- o orçamento de consultas ao LLM da tese, reduzido na proporção do tamanho desta base;
- orçamento humano ilimitado.

A política não vê PAR ao decidir; PAR entra depois, para simular as respostas do LLM e contar acertos e falsos positivos.

**Simplificações frente à tese.** Na tese, o arranjo que preserva o corte e decide abaixo dele pela política de quatro ações, com revisão assistida por LLM, chama-se ASR. Aqui a política usa a mesma probabilidade de O3, enquanto ASR usou uma probabilidade recalibrada (O4). E os `MATCH` automáticos entram como vínculo sem revisão humana e não contam na carga humana, o que favorece a política na comparação.

In [ ]:
assert "base" in globals(), (
    "A sessão perdeu as variáveis das seções anteriores (foi reiniciada?). "
    "Clique nesta célula e use Ambiente de execução > Executar antes (Ctrl+F8)."
)
t0 = time.perf_counter()
politicas = {modo: nf.politica_v31(abaixo, p_oof, modo, seed=SEED) for modo in nf.CUSTOS_POR_MODO}
print(f"Política v3.1 nos dois modos: {br(time.perf_counter() - t0, 1)} s.")
resumos = pd.DataFrame({modo: resumo for modo, (_, resumo) in politicas.items()})
mostrar(resumos.rename(columns=NOMES_MODO))
aceitos = resumos.loc["aceitos automaticamente (MATCH)"]
print(f"Aceitos automaticamente: {inteiro(aceitos['vigilancia'])} na vigilância e "
      f"{inteiro(aceitos['confirmacao'])} na confirmação.")
confere("a vigilância aceita automaticamente pelo menos tantos potenciais pares quanto a confirmação",
        aceitos["vigilancia"] >= aceitos["confirmacao"])

**Leitura.** Cada coluna é um modo. As linhas contam o que a política fez com os potenciais pares abaixo do corte: quantos aceitou ou rejeitou sozinha, quantos mandou ao revisor humano e quantos ao LLM, e o que o LLM respondeu. A vigilância aceita pelo menos tanto quanto a confirmação (conferido acima): nela o falso positivo custa menos, e aceitar compensa com probabilidades menores. Repare nas linhas de falsos positivos: a política os produz; a fila, com revisor perfeito, não.

**Na mesma carga humana.** Para comparar com justiça, contamos quantas revisões humanas a política gasta e damos às filas exatamente essa carga. A política tem duas leituras:

- **o LLM aceita o `MATCH`**: o `MATCH` do LLM vira vínculo sem revisor humano;
- **o `MATCH` do LLM vai ao humano**: variante que a tese indica caso se use o LLM, ainda como hipótese exploratória; o `MATCH` do LLM só indica o potencial par ao revisor humano.

A função `mesma_carga` dá a cada fila a carga da variante e compara pares verdadeiros recuperados e falsos positivos. A última coluna diz quantas revisões cada fila precisaria para igualar a variante. A célula imprime, por modo, os falsos positivos do próprio LLM e o total quando ele aceita o `MATCH` sozinho (somando os `MATCH` automáticos).

In [ ]:
comparacoes = {}
for modo, (decisoes, _) in politicas.items():
    tabela = nf.mesma_carga(abaixo, ordens, decisoes).set_index("estratégia")
    tabela["revisões para igualar a variante"] = tabela["revisões para igualar a variante"].astype("Int64")
    comparacoes[modo] = tabela
    print(f"Modo {NOMES_MODO[modo]}:")
    mostrar(tabela)

for modo, tabela in comparacoes.items():
    variante = tabela.loc["v3.1, MATCH do LLM ao humano"]
    carga, rec_var = int(variante["revisões humanas"]), int(variante["pares verdadeiros recuperados"])
    rec_filas = {fila: int(tabela.loc[f"fila {fila}", "pares verdadeiros recuperados"]) for fila in ordens}
    fp_llm = int(resumos.loc["falsos positivos do LLM", modo])
    fp_total = int(tabela.loc["v3.1, LLM aceita o MATCH", "falsos positivos"])
    print(f"{NOMES_MODO[modo]}: com {inteiro(carga)} revisões humanas, a v3.1 recupera {inteiro(rec_var)} "
          f"PAR = 2; O3, {inteiro(rec_filas['O3'])}; F4, {inteiro(rec_filas['F4'])}.")
    print(f"   Se o LLM aceitasse o MATCH sozinho: {inteiro(fp_llm)} falsos positivos do LLM e "
          f"{inteiro(fp_total)} no total, somando os MATCH automáticos.")
    diferenca_o3 = rec_var - rec_filas["O3"]
    if diferenca_o3 <= 0:
        print("   Leitura: a automação não recupera mais que a fila O3 na mesma carga.")
    else:
        print(f"   Leitura: a automação recupera {inteiro(diferenca_o3)} PAR = 2 a mais que a fila O3 "
              f"na mesma carga; veja quantas revisões a fila precisaria para igualá-la.")

rec_variante_vig = int(comparacoes["vigilancia"].loc["v3.1, MATCH do LLM ao humano", "pares verdadeiros recuperados"])
rec_o3_vig = int(comparacoes["vigilancia"].loc["fila O3", "pares verdadeiros recuperados"])
confere("na vigilância, aceitar o MATCH do LLM sem revisor humano acrescenta falsos positivos",
        resumos.loc["falsos positivos do LLM", "vigilancia"] > 0)
confere("na vigilância, na mesma carga humana, a automação recupera no máximo um PAR = 2 a mais que O3",
        rec_variante_vig - rec_o3_vig <= 1)

**Leitura.** Na vigilância, na mesma carga humana, a automação recupera no máximo um PAR = 2 a mais que a fila O3 (conferido acima): não reduz a revisão manual além do que a ordenação já reduz, mesmo com a simplificação que a favorece. Na confirmação, leia a diferença impressa e a coluna de revisões para igualar a variante, que diz quanto esforço a mais a fila precisaria. Quando o LLM aceita o `MATCH` sozinho, ele acrescenta falsos positivos próprios (conferido na vigilância; na confirmação, leia a contagem impressa). A célula seguinte mede a proporção de pares verdadeiros entre as respostas `MATCH` do LLM simulado, isto é, o VPP dessas respostas.

In [ ]:
vpp_llm = {}
for modo, (decisoes, _) in politicas.items():
    match_llm = decisoes["resposta_llm"].eq("MATCH").to_numpy()
    verdadeiro = nf.rotulo(decisoes).astype(bool)
    n_match, acertos = int(match_llm.sum()), int((match_llm & verdadeiro).sum())
    vpp_llm[modo] = acertos / n_match if n_match else float("nan")
    if n_match:
        print(f"{NOMES_MODO[modo]}: o LLM simulado respondeu MATCH {inteiro(n_match)} vezes; "
              f"{inteiro(acertos)} eram pares verdadeiros (VPP = {pct(vpp_llm[modo])}).")
    else:
        print(f"{NOMES_MODO[modo]}: o LLM simulado não respondeu MATCH nesta execução.")
confere("na vigilância, menos da metade das respostas MATCH do LLM simulado são pares verdadeiros",
        vpp_llm["vigilancia"] < 0.5)

**Leitura.** Mesmo com taxas de falso positivo pequenas, o LLM é aplicado a muitos não pares e poucos pares verdadeiros: os falsos positivos acumulam e o VPP do `MATCH` fica baixo. Por isso, na tese, o LLM só apoia a revisão: o `MATCH` dele segue ao revisor humano e nunca vira vínculo sozinho.

## 9. Linha de base histórica: o perfil v3 reproduzível

> **Seção técnica, pode pular.** Ela confere que o código do perfil histórico continua o mesmo; não acrescenta nada à recomendação.

O pacote mantém o **perfil histórico** (`run_v3` com `p_cal="platt"` e `policy_version="v3"`), porque a tese compara o antes e o depois. Nesse perfil:

- a probabilidade vem só da nota final, por *Platt scaling*, uma regressão logística de uma variável, ajustada e aplicada nas mesmas linhas (dentro da amostra);
- a política v3 tem três ações (`MATCH`, `NONMATCH`, `LLM_REVIEW`) e guardas próprias;
- a decisão vale para a base inteira, sem preservar o ponto de corte.

Este perfil **não é a recomendação**. Ele usa a base sintética antiga (`synthetic_data`), que tem as colunas que esse perfil espera (inclusive as datas de que as MACD precisam) e não tem escore do OpenRecLink nem ponto de corte. A primeira célula importa as funções do pacote usadas nesta seção.

In [ ]:
assert "base" in globals(), (
    "A sessão perdeu as variáveis das seções anteriores (foi reiniciada?). "
    "Clique nesta célula e use Ambiente de execução > Executar antes (Ctrl+F8)."
)
from dataclasses import replace

import synthetic_data
from gzcmd_record_linkage.bands import BandAssigner
from gzcmd_record_linkage.calibration import compute_p_cal, fit_calibration_from_df
from gzcmd_record_linkage.guardrails import apply_guardrails
from gzcmd_record_linkage.loader import LoadConfig, load_comparador_csv
from gzcmd_record_linkage.runner import build_engine_from_config, run_v3

print(f"Pasta da base antiga: {DATA_DIR}")

**Célula técnica, pode pular: a base antiga, conferida.** A célula gera a base antiga com a mesma semente e a compara com o CSV versionado em `data/synthetic/`. Se o arquivo existe, ele não é sobrescrito: só conferido. Se não existe, é criado.

In [ ]:
dataset = synthetic_data.generate_comparador(seed=SEED, n_pairs=600, scenarios="all")
df_hist = dataset.frame
CSV_HIST = DATA_DIR / "comparador_sintetico.csv"
if CSV_HIST.exists():
    import tempfile

    with tempfile.TemporaryDirectory() as pasta_temporaria:
        esperado = Path(pasta_temporaria) / CSV_HIST.name
        synthetic_data.to_comparador_csv(df_hist, esperado)
        assert esperado.read_text(encoding="utf-8") == CSV_HIST.read_text(encoding="utf-8"), (
            "O CSV versionado difere dos dados gerados. Confira a versão do gerador."
        )
    print("CSV versionado conferido e preservado.")
else:
    synthetic_data.to_comparador_csv(df_hist, CSV_HIST)
    print("CSV ausente: arquivo criado.")
proporcao_hist = f"{100 * dataset.meta['match_ratio_realized']:.1f}".replace(".", ",")
print(f"Potenciais pares da base antiga: {len(df_hist)}")
print(f"Proporção de pares verdadeiros: {proporcao_hist}%")

**O perfil histórico pelo `run_v3`.** A célula roda o perfil nos dois modos e conta as ações. O resumo traz também os dois parâmetros ajustados pelo *Platt scaling* (intercepto e inclinação).

In [ ]:
historico, resumos_hist = {}, {}
for modo in nf.CUSTOS_POR_MODO:
    historico[modo], resumos_hist[modo] = run_v3(
        input_csv=CSV_HIST, config_path=CONFIG_PATH, mode=modo,
        macd_enabled=True, p_cal="platt", policy_version="v3",
    )
acoes_hist = pd.DataFrame({modo: saida["action"].value_counts() for modo, saida in historico.items()})
mostrar(acoes_hist.fillna(0).astype(int).rename(columns=NOMES_MODO))
for modo, resumo_run in resumos_hist.items():
    print(f"{NOMES_MODO[modo]}: perfil {resumo_run.policy_version}; Platt scaling com intercepto "
          f"{br(resumo_run.p_cal_params['intercept'], 4)} e inclinação "
          f"{br(resumo_run.p_cal_params['slope'], 4)}; consultas ao LLM: "
          f"{inteiro(resumo_run.llm_used)} de {inteiro(resumo_run.llm_max)}.")

**Célula técnica, pode pular: reconciliação passo a passo.** Refazemos o perfil de vigilância com as funções do pacote, uma etapa por vez (carga, faixas da nota final, *Platt scaling*, guardas v3, motor v3), e conferimos que o resultado é idêntico, coluna a coluna, ao do `run_v3`.

In [ ]:
df_passos = load_comparador_csv(CSV_HIST, cfg=LoadConfig(macd_enabled=True))
df_passos["band"] = BandAssigner.from_config(cfg).assign(df_passos["nota_final"])
platt_hist = fit_calibration_from_df(df_passos, config=replace(cfg.calibration, method="platt"))
df_passos["p_cal"] = compute_p_cal(
    df_passos, method="platt", model=platt_hist,
    clip_min=float(cfg.calibration.clip_min), clip_max=float(cfg.calibration.clip_max),
)
guardas_hist = apply_guardrails(
    df_passos,
    temporal_days=cfg.guardrails.temporal_days,
    nota_always_match=cfg.guardrails.nota_always_match,
    nota_always_nonmatch=cfg.guardrails.nota_always_nonmatch,
    homonimia_min_nota=cfg.guardrails.homonimia_min_nota,
    homonimia_year_gap=cfg.guardrails.homonimia_year_gap,
    policy_version="v3",
)
df_passos["guardrail"] = guardas_hist.guardrail
manual = build_engine_from_config(cfg, mode="vigilancia", policy_version="v3").triage(df_passos.copy())
for coluna in ["band", "p_cal", "guardrail", "base_choice", "evr", "action"]:
    pd.testing.assert_series_equal(manual[coluna], historico["vigilancia"][coluna], check_exact=True)
    print(f"✓ {coluna}: idêntico ao run_v3")

**Leitura.** A reprodução coincide coluna a coluna: o perfil histórico continua o mesmo, e a comparação de antes e depois da tese pode ser refeita. A igualdade confere o código, não o desempenho: o *Platt scaling* é ajustado e aplicado nas mesmas linhas, e a política decide na base inteira, sem preservar o corte. A recomendação é a das seções 4 a 6.

## 10. Fluxo recomendado e limitações

### 10.1 O que a tese recomenda

As citações e os números desta seção vêm do capítulo de conclusão da tese (`conclusao.tex`), identificados pelo rótulo da seção, na versão pública: [release tese-2026-11-11-7e91702](https://github.com/marco-jardim/phd-research/releases/tag/tese-2026-11-11-7e91702). Nelas, F3 é a fusão de O3, de O1 e de uma terceira fila, a da probabilidade calibrada com as MACD; e **referência cega** é uma revisão manual feita sem acesso à nota final e aos subescores, como a tese pede para a validação em coorte nova.

Na seção Implicações para o serviço (`sec:concl-recomendacoes`):

> O fluxo recomendado, a validar em coorte nova, começa no OpenRecLink, que gera os potenciais pares pela blocagem e aceita os de escore no ponto de corte ou acima dele. Abaixo dele, o CdR calcula os subescores dos campos de identificação, o XGBoost os converte, com a nota final, em probabilidade calibrada fora da amostra (O3), e a revisão manual segue a fila ordenada por essa probabilidade, isolada ou fundida ao escore do OpenRecLink, até o orçamento humano disponível. Frente ao escore isolado (O1), as fusões F3 e F4 exigiram menos revisões nas metas de 9 e de 36 pares verdadeiros, com IC abaixo de zero, e O3 isolada, só pontualmente nas metas menores, exigindo mais revisões que O1 na recuperação integral; nenhuma fusão superou O3 com IC, e a curva de esforço indica qual ordenação usar. O LLM pode apoiar a revisão, mas não deve aceitar vínculos sozinho.

E, na mesma seção:

> Recomenda-se, a validar em coorte nova com referência cega, preservar o corte do OpenRecLink e, abaixo dele, ordenar a fila de revisão manual pela probabilidade calibrada fora da amostra, isolada (O3) ou fundida ao escore do OpenRecLink por posição recíproca (F4; F3 quando houver as MACD), escolhendo entre elas, e entre estimadores, pela curva de esforço. Com o orçamento de 17 revisões da confirmação, O3 recupera 6 pares verdadeiros; O2, ordenação pela nota final do CdR, 9; e F4, 16 (`data/comparador_mesma_carga.csv`, para O2 e O3, e `data/fusao_filas.csv`, para F4).

Os caminhos `data/*.csv` da citação são arquivos do repositório da tese (`phd-research`), não deste pacote.

Em termos deste notebook, o fluxo é:

1. preservar os vínculos aceitos pelo ponto de corte (seção 3);
2. abaixo dele, estimar a probabilidade calibrada fora da amostra, com partição por óbito (seção 4);
3. traçar a curva de esforço de O3 e F4 numa amostra revisada e escolher a fila e o ponto de parada pelo orçamento humano (seções 5 e 6);
4. medir a incerteza com o *bootstrap* agrupado por óbito (seção 7);
5. se houver LLM, usá-lo só como apoio: o `MATCH` dele segue ao revisor humano (seção 8).

**Onde está cada peça.** Neste notebook, as funções vêm de `nb_fila.py`, na pasta `notebooks/`, fora do pacote publicado: `prob_calibrada_oof`, `ordenacoes`, `curva_esforco`, `revisoes_para_meta`, `bootstrap_diferenca`, `politica_v31` e `mesma_carga`. No pacote `gzcmd_record_linkage`, a fusão está em `gzcmd_record_linkage.ranking`: `rrf` e `review_queue(df, scores=[...], method="rrf")`, que ordena uma fila já selecionada abaixo do corte pelas colunas indicadas (maior primeiro). Nos dois, os empates são desfeitos por uma permutação aleatória com semente fixa sobre a posição das linhas, nunca pela classe: mudar a semente ou a ordem das linhas muda a ordem dentro dos empates e, com ela, o número de revisões, sobretudo na recuperação integral.

**Consequência para a vigilância.** Os potenciais pares abaixo do ponto de parada não foram examinados e devem ser registrados como não examinados, não como não pares: na vigilância do óbito, a ausência de vínculo por esgotamento do orçamento não equivale a óbito não notificado (`sec:concl-recomendacoes`).

### 10.2 Números da tese, como referência citada

Os números abaixo são da coorte real da tese (SIM e Sinan, tuberculose), não desta base sintética. Todos supõem revisor perfeito simulado e vêm de análise *post hoc*. As metas da seção 6 (25%, 64% e 100% dos PAR = 2) seguem as frações das metas da tese, que tinha 36 pares verdadeiros abaixo do corte. **ASR** é o arranjo da tese que preserva o corte e decide abaixo dele pela política de quatro ações, com revisão assistida por LLM.

| Resultado na tese | Valor | Seção (`conclusao.tex`) |
|---|---|---|
| Pares verdadeiros na coorte | 247, dos quais 36 abaixo do corte | `sec:concl-limitacoes` |
| Revisões para recuperar os 23 pares verdadeiros que ASR recupera na vigilância | O3: 125; O1: 501 | `sec:concl-sintese` |
| Revisões para recuperar os 9 que ASR recupera na confirmação | O3: 29; O1: 51 | `sec:concl-sintese` |
| Revisões para recuperar os 36 pares verdadeiros abaixo do corte | F4: 1.387; O1: 2.521; O3: 23.443 | `sec:concl-sintese` |
| *Bootstrap* agrupado, O3 menos O1 | nenhum IC 95% inteiramente abaixo de zero, nas metas de 9, 23 e 36 | `sec:concl-sintese` |
| *Bootstrap* agrupado, F4 menos O1 | IC 95% inteiramente abaixo de zero nas metas de 9 (mediana de 38 revisões a menos) e de 36 (1.203 a menos); não na de 23 | `sec:concl-sintese` |
| Fusões frente a O3 | nenhuma fusão superou O3 com IC | `sec:concl-recomendacoes` |
| Mesma carga humana na vigilância, 96 revisões | ASR: 23 pares verdadeiros, com 21 falsos positivos; O3: 22; "diferença que não sustenta superioridade" | abertura do capítulo (`cap:conclusao`) e `sec:concl-sintese` |
| VPP das aceitações do LLM abaixo do corte, na vigilância | 16,0% (IC 95% de 6,4% a 34,7%); estaria subestimado se algum dos falsos positivos situados em região sem revisão documentada fosse par verdadeiro não verificado | `sec:concl-sintese` e `sec:concl-recomendacoes` |

As leituras que as seções 6 a 8 conferem nesta base (nenhuma fila domina em todas as metas; O3 exige menos revisões que O1 nas metas menores; aceitar o `MATCH` do LLM acrescenta falsos positivos; o VPP desse `MATCH` é baixo) lembram as da tese. A base, porém, foi ajustada para imitar a dificuldade do problema, não os resultados, e os seus valores não se comparam aos da tabela.

### 10.3 Limitações

- **Dados sintéticos.** A base imita a dificuldade da tese, não seus números. Os mecanismos (registros precários, homônimos, passo 11 fraco, viés de verificação) foram escolhidos por quem escreveu o gerador; resultados aqui demonstram o método, não o validam.
- **Revisor perfeito simulado.** Todas as contagens de revisões supõem que o revisor humano acerta sempre; a sensibilidade humana real não foi medida, nem aqui nem na tese (`sec:concl-limitacoes`).
- **Análise *post hoc* e escolha de F4.** Na tese, as comparações entre filas, o *bootstrap* e a escolha de F4 entre seis variantes, avaliadas nos mesmos 36 pares verdadeiros, são posteriores ao registro; não há correção para multiplicidade, e a fixação anterior ao cálculo não constitui pré-registro (`sec:concl-limitacoes`).
- **Referência operacional.** Na tese, a revisão manual de origem foi feita por revisora única, sem classificação independente. Há viés de verificação, que favorece as filas que usam o escore (O1 e F4), e risco de viés de incorporação, que favoreceria O2 e O3; sem suposições sobre o mecanismo de verificação, a direção dominante e a magnitude do viés não são identificáveis (`sec:concl-limitacoes`). Na meta de 36, parte do ganho de F4 vem de priorizar a faixa de maior escore, que contém todos os pares verdadeiros recuperados pela revisão de origem (`sec:concl-sintese`). A simulação reproduz o viés de verificação de propósito, e a avaliação aqui também usa só PAR.
- **LLM simulado.** As respostas vêm de taxas de erro por faixa do revisor de referência da tese (Claude Haiku 5.5), sorteadas de modo independente para cada potencial par. Nenhum LLM foi consultado, e nada aqui mede um LLM real.
- **Incerteza condicional.** Neste notebook, o *bootstrap* não refaz a fusão (na tese, a fusão foi refeita em cada réplica); nos dois, o modelo não é reajustado e a referência é tomada como certa.
- **Perfil histórico.** A seção 9 confere a reprodução do código, não o desempenho do perfil v3.

### 10.4 Para levar

A recomendação não é um algoritmo que decide sozinho, e sim uma **ordem de trabalho** para o revisor humano: manter o que o corte já aceitou, ordenar a *grey area* pela probabilidade calibrada, isolada ou fundida ao escore, e parar onde o orçamento humano mandar, lendo a curva de esforço. Na tese, a automação e o LLM, nas condições avaliadas, não reduziram a revisão manual além dessa ordenação.